In [ ]:
# Celda 1: Leer Excel y generar CSV resumen de hojas/columnas

import pandas as pd
from pathlib import Path
import os

# Ruta de la carpeta donde está el Excel
carpeta_data = Path(
    r"C:\Users\isancheza\OneDrive - SBS\Documentos\UPC\Data Vizualitation\7304416-base-de-datos-sidpol-a-setiembre-2025(2) (1)\Data"
)

# Buscar automáticamente el archivo Excel de SIDPOL
archivos_excel = list(carpeta_data.glob("*.xlsx"))

archivo_excel = None
for archivo in archivos_excel:
    if "SIDPOL" in archivo.name.upper():
        archivo_excel = archivo
        break

if archivo_excel is None:
    raise FileNotFoundError("No se encontró un archivo Excel que contenga 'SIDPOL' en el nombre.")

# Crear carpeta de salida
carpeta_salida = carpeta_data / "salida_revision"
carpeta_salida.mkdir(exist_ok=True)

# Leer archivo Excel
excel = pd.ExcelFile(archivo_excel)

# Diccionario para guardar cada hoja leída
diccionario_hojas = {}

# Lista para construir el resumen
resumen_columnas = []

for hoja in excel.sheet_names:
    df_temp = pd.read_excel(archivo_excel, sheet_name=hoja)
    diccionario_hojas[hoja] = df_temp
    
    total_filas = df_temp.shape[0]
    total_columnas = df_temp.shape[1]
    
    for orden_columna, columna in enumerate(df_temp.columns, start=1):
        serie = df_temp[columna]
        
        valores_ejemplo = (
            serie
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(5)
            .tolist()
        )
        
        resumen_columnas.append({
            "hoja": hoja,
            "filas_hoja": total_filas,
            "columnas_hoja": total_columnas,
            "orden_columna": orden_columna,
            "nombre_columna": columna,
            "tipo_dato": str(serie.dtype),
            "valores_nulos": int(serie.isna().sum()),
            "porcentaje_nulos": round(serie.isna().mean() * 100, 2),
            "valores_unicos": int(serie.nunique(dropna=True)),
            "ejemplos_valores": " | ".join(valores_ejemplo)
        })

# Convertir resumen a DataFrame
df_resumen_columnas = pd.DataFrame(resumen_columnas)

# Guardar CSV resumen
ruta_resumen_csv = carpeta_salida / "resumen_columnas_por_hoja.csv"

df_resumen_columnas.to_csv(
    ruta_resumen_csv,
    index=False,
    encoding="utf-8-sig"
)

# Abrir automáticamente el CSV en Windows
os.startfile(ruta_resumen_csv)

In [ ]:
# Celda 2: Limpieza general de todas las hojas SIDPOL

import pandas as pd
import numpy as np
from pathlib import Path
import re
import unicodedata
import os
from functools import reduce

# =========================
# 1. Rutas
# =========================

carpeta_data = Path(
    r"C:\Users\isancheza\OneDrive - SBS\Documentos\UPC\Data Vizualitation\7304416-base-de-datos-sidpol-a-setiembre-2025(2) (1)\Data"
)

archivos_excel = list(carpeta_data.glob("*.xlsx"))

archivo_excel = None
for archivo in archivos_excel:
    if "SIDPOL" in archivo.name.upper():
        archivo_excel = archivo
        break

if archivo_excel is None:
    raise FileNotFoundError("No se encontró el archivo Excel de SIDPOL.")

carpeta_salida = carpeta_data / "salida_limpieza_sidpol"
carpeta_salida.mkdir(exist_ok=True)

carpeta_hojas_limpias = carpeta_salida / "hojas_limpias"
carpeta_hojas_limpias.mkdir(exist_ok=True)

# =========================
# 2. Funciones de limpieza
# =========================

def quitar_tildes(texto):
    if pd.isna(texto):
        return np.nan
    
    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    return texto


def limpiar_texto(texto):
    """
    Normaliza textos:
    - quita tildes
    - convierte a mayúsculas
    - elimina espacios dobles
    - limpia espacios al inicio y final
    """
    if pd.isna(texto):
        return np.nan
    
    texto = quitar_tildes(texto)
    texto = texto.upper()
    texto = texto.strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto


def limpiar_nombre_columna(columna):
    """
    Normaliza nombres de columnas.
    """
    columna = quitar_tildes(columna)
    columna = columna.upper().strip()
    columna = re.sub(r"\s+", "_", columna)
    columna = columna.replace(".", "_")
    columna = columna.replace("-", "_")
    columna = re.sub(r"_+", "_", columna)
    return columna


def normalizar_ubigeo(valor):
    """
    Convierte UBIGEO a texto de 6 dígitos.
    Ejemplo:
    10101 -> 010101
    """
    if pd.isna(valor):
        return np.nan
    
    try:
        valor = str(int(float(valor)))
    except:
        valor = str(valor).strip()
    
    valor = re.sub(r"\D", "", valor)
    
    if valor == "":
        return np.nan
    
    return valor.zfill(6)


def crear_fecha(anio, mes):
    """
    Crea fecha usando año y mes.
    Día fijo = 1.
    """
    return pd.to_datetime(
        dict(year=anio, month=mes, day=1),
        errors="coerce"
    )


def limpiar_hoja_sidpol(df, nombre_hoja):
    """
    Limpia una hoja del Excel SIDPOL.
    """
    
    df = df.copy()
    
    # Normalizar nombres de columnas
    df.columns = [limpiar_nombre_columna(c) for c in df.columns]
    
    # Renombrar columnas a nombres estándar
    mapa_columnas = {
        "ANIO": "anio",
        "MES": "mes",
        "DPTO_HECHO_NEW": "departamento",
        "PROV_HECHO": "provincia",
        "DIST_HECHO": "distrito",
        "UBIGEO_HECHO": "ubigeo",
        "N_DIST_ID_DGC": "conteo",
        "ES_DELITO_X": "es_delito",
        "PRINCIPALES_TIPOS": "principales_tipos",
        "PMODALIDADES": "p_modalidad",
        "P_MODALIDADES": "p_modalidad",
        "TIPO": "tipo",
        "SUB_TIPO": "sub_tipo",
        "MODALIDAD": "modalidad",
        "DIST_EMERGENCIA": "dist_emergencia"
    }
    
    df = df.rename(columns=mapa_columnas)
    
    # Agregar hoja de origen
    df["fuente_hoja"] = nombre_hoja
    
    # Convertir año y mes
    if "anio" in df.columns:
        df["anio"] = pd.to_numeric(df["anio"], errors="coerce").astype("Int64")
    
    if "mes" in df.columns:
        df["mes"] = pd.to_numeric(df["mes"], errors="coerce").astype("Int64")
    
    # Convertir conteo
    if "conteo" in df.columns:
        df["conteo"] = pd.to_numeric(df["conteo"], errors="coerce").fillna(0).astype(int)
    
    # Normalizar UBIGEO
    if "ubigeo" in df.columns:
        df["ubigeo"] = df["ubigeo"].apply(normalizar_ubigeo)
    
    # Normalizar textos
    columnas_texto = [
        "departamento",
        "provincia",
        "distrito",
        "es_delito",
        "principales_tipos",
        "p_modalidad",
        "tipo",
        "sub_tipo",
        "modalidad"
    ]
    
    for col in columnas_texto:
        if col in df.columns:
            df[col] = df[col].apply(limpiar_texto)
    
    # Crear fecha
    if "anio" in df.columns and "mes" in df.columns:
        df["fecha"] = crear_fecha(df["anio"], df["mes"])
        df["trimestre"] = df["fecha"].dt.quarter.astype("Int64")
        df["anio_mes"] = df["fecha"].dt.strftime("%Y-%m")
    
    # Crear nivel geográfico
    if "ubigeo" in df.columns:
        df["nivel_geografico"] = "DISTRITO"
    else:
        df["nivel_geografico"] = "DEPARTAMENTO"
    
    # Crear columnas vacías si no existen para estandarizar estructura
    columnas_base = [
        "fuente_hoja",
        "nivel_geografico",
        "anio",
        "mes",
        "fecha",
        "anio_mes",
        "trimestre",
        "ubigeo",
        "departamento",
        "provincia",
        "distrito",
        "dist_emergencia",
        "es_delito",
        "principales_tipos",
        "p_modalidad",
        "tipo",
        "sub_tipo",
        "modalidad",
        "conteo"
    ]
    
    for col in columnas_base:
        if col not in df.columns:
            df[col] = np.nan
    
    # Ordenar columnas
    df = df[columnas_base]
    
    # Eliminar filas sin año, mes o conteo
    df = df.dropna(subset=["anio", "mes"])
    df = df[df["conteo"] >= 0]
    
    return df


def eliminar_duplicados_sin_sumar(df):
    """
    Elimina duplicados conservando el mayor conteo.
    No suma conteos.
    """
    
    columnas_llave = [c for c in df.columns if c != "conteo"]
    
    df_limpio = (
        df
        .groupby(columnas_llave, dropna=False, as_index=False)["conteo"]
        .max()
    )
    
    return df_limpio


# =========================
# 3. Leer, limpiar y guardar cada hoja
# =========================

excel = pd.ExcelFile(archivo_excel)

hojas_limpias = {}
qa_limpieza = []

for hoja in excel.sheet_names:
    df_original = pd.read_excel(archivo_excel, sheet_name=hoja)
    
    filas_originales = len(df_original)
    
    df_limpio = limpiar_hoja_sidpol(df_original, hoja)
    filas_despues_limpieza = len(df_limpio)
    
    df_limpio_sin_duplicados = eliminar_duplicados_sin_sumar(df_limpio)
    filas_finales = len(df_limpio_sin_duplicados)
    
    duplicados_eliminados = filas_despues_limpieza - filas_finales
    
    hojas_limpias[hoja] = df_limpio_sin_duplicados
    
    # Guardar cada hoja limpia en CSV
    nombre_archivo_csv = f"{hoja.replace('.', '_')}_limpia.csv"
    ruta_csv = carpeta_hojas_limpias / nombre_archivo_csv
    
    df_limpio_sin_duplicados.to_csv(
        ruta_csv,
        index=False,
        encoding="utf-8-sig"
    )
    
    qa_limpieza.append({
        "hoja": hoja,
        "filas_originales": filas_originales,
        "filas_despues_limpieza": filas_despues_limpieza,
        "filas_finales_sin_duplicados": filas_finales,
        "duplicados_eliminados": duplicados_eliminados,
        "columnas_finales": len(df_limpio_sin_duplicados.columns),
        "nivel_geografico": df_limpio_sin_duplicados["nivel_geografico"].dropna().unique()[0],
        "anio_min": df_limpio_sin_duplicados["anio"].min(),
        "anio_max": df_limpio_sin_duplicados["anio"].max(),
        "conteo_total_referencial": df_limpio_sin_duplicados["conteo"].sum()
    })

df_qa_limpieza = pd.DataFrame(qa_limpieza)

ruta_qa = carpeta_salida / "qa_limpieza_por_hoja.csv"

df_qa_limpieza.to_csv(
    ruta_qa,
    index=False,
    encoding="utf-8-sig"
)

os.startfile(carpeta_salida)

In [ ]:
# Celda 3: Crear tabla larga unificada

def construir_tabla_larga(hojas_limpias):
    tablas = []
    
    for hoja, df in hojas_limpias.items():
        df = df.copy()
        
        # Temp2: clasificación general ES_DELITO
        if hoja == "Temp2":
            temp = df.copy()
            temp["nivel_categoria"] = "ES_DELITO"
            temp["categoria_analitica"] = temp["es_delito"]
            tablas.append(temp)
        
        # Temp3 y Temp5: principales tipos
        elif hoja in ["Temp3", "Temp5"]:
            temp = df.copy()
            temp["nivel_categoria"] = "PRINCIPALES_TIPOS"
            temp["categoria_analitica"] = temp["principales_tipos"]
            tablas.append(temp)
        
        # Temp4 y Temp5.2: principales modalidades
        elif hoja in ["Temp4", "Temp5.2"]:
            temp = df.copy()
            temp["nivel_categoria"] = "P_MODALIDAD"
            temp["categoria_analitica"] = temp["p_modalidad"]
            tablas.append(temp)
        
        # Temp6 y Temp7: detalle completo tipo, subtipo y modalidad
        elif hoja in ["Temp6", "Temp7"]:
            temp = df.copy()
            temp["nivel_categoria"] = "DETALLE_TIPO_SUBTIPO_MODALIDAD"
            temp["categoria_analitica"] = temp["modalidad"]
            tablas.append(temp)
    
    df_largo = pd.concat(tablas, ignore_index=True)
    
    columnas_finales = [
        "fuente_hoja",
        "nivel_geografico",
        "nivel_categoria",
        "anio",
        "mes",
        "fecha",
        "anio_mes",
        "trimestre",
        "ubigeo",
        "departamento",
        "provincia",
        "distrito",
        "dist_emergencia",
        "es_delito",
        "principales_tipos",
        "p_modalidad",
        "tipo",
        "sub_tipo",
        "modalidad",
        "categoria_analitica",
        "conteo"
    ]
    
    df_largo = df_largo[columnas_finales]
    
    # Eliminar duplicados exactos en la tabla larga conservando mayor conteo
    columnas_llave = [c for c in df_largo.columns if c != "conteo"]
    
    df_largo = (
        df_largo
        .groupby(columnas_llave, dropna=False, as_index=False)["conteo"]
        .max()
    )
    
    return df_largo


df_sidpol_largo = construir_tabla_larga(hojas_limpias)

ruta_largo = carpeta_salida / "sidpol_largo_unificado_limpio.csv"

df_sidpol_largo.to_csv(
    ruta_largo,
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Celda 4: Crear tabla ancha tipo super merge sin multiplicar filas

def nombre_seguro_columna(texto):
    """
    Convierte una categoría en nombre de columna seguro.
    """
    if pd.isna(texto):
        return "SIN_CATEGORIA"
    
    texto = limpiar_texto(texto)
    texto = re.sub(r"[^A-Z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto)
    texto = texto.strip("_")
    
    if texto == "":
        texto = "SIN_CATEGORIA"
    
    return texto


def crear_pivot_seguro(df, columna_categoria, prefijo):
    """
    Crea una tabla ancha por distrito-mes.
    Usa max para evitar sumar duplicados.
    """
    
    df = df.copy()
    df = df[df["nivel_geografico"] == "DISTRITO"]
    
    df = df.dropna(subset=["ubigeo", columna_categoria])
    
    if df.empty:
        return pd.DataFrame()
    
    llaves = [
        "anio",
        "mes",
        "fecha",
        "anio_mes",
        "trimestre",
        "ubigeo",
        "departamento",
        "provincia",
        "distrito"
    ]
    
    if "dist_emergencia" in df.columns:
        # Se agrega después como max por llave
        pass
    
    df["categoria_columna"] = (
        prefijo
        + "__"
        + df[columna_categoria].apply(nombre_seguro_columna)
    )
    
    pivot = (
        df
        .pivot_table(
            index=llaves,
            columns="categoria_columna",
            values="conteo",
            aggfunc="max",
            fill_value=0
        )
        .reset_index()
    )
    
    pivot.columns.name = None
    
    return pivot


# =========================
# 1. Base geográfica distrito-mes
# =========================

df_distrito = pd.concat(
    [
        hojas_limpias[h]
        for h in hojas_limpias.keys()
        if "ubigeo" in hojas_limpias[h].columns
    ],
    ignore_index=True
)

df_distrito = df_distrito[df_distrito["nivel_geografico"] == "DISTRITO"]

llaves_base = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "trimestre",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito"
]

df_base_distrito_mes = (
    df_distrito[llaves_base + ["dist_emergencia"]]
    .drop_duplicates()
    .groupby(llaves_base, dropna=False, as_index=False)["dist_emergencia"]
    .max()
)

# =========================
# 2. Pivots por tipo de información
# =========================

tablas_pivot = []

# Principales tipos: Temp5
if "Temp5" in hojas_limpias:
    pivot_pt = crear_pivot_seguro(
        hojas_limpias["Temp5"],
        columna_categoria="principales_tipos",
        prefijo="PT"
    )
    if not pivot_pt.empty:
        tablas_pivot.append(pivot_pt)

# Principales modalidades: Temp5.2
if "Temp5.2" in hojas_limpias:
    pivot_pm = crear_pivot_seguro(
        hojas_limpias["Temp5.2"],
        columna_categoria="p_modalidad",
        prefijo="PM"
    )
    if not pivot_pm.empty:
        tablas_pivot.append(pivot_pm)

# Tipo detallado: Temp6 y Temp7
df_detalle = pd.concat(
    [
        hojas_limpias[h]
        for h in ["Temp6", "Temp7"]
        if h in hojas_limpias
    ],
    ignore_index=True
)

if not df_detalle.empty:
    pivot_tipo = crear_pivot_seguro(
        df_detalle,
        columna_categoria="tipo",
        prefijo="TIPO"
    )
    if not pivot_tipo.empty:
        tablas_pivot.append(pivot_tipo)

    pivot_subtipo = crear_pivot_seguro(
        df_detalle,
        columna_categoria="sub_tipo",
        prefijo="SUBTIPO"
    )
    if not pivot_subtipo.empty:
        tablas_pivot.append(pivot_subtipo)

    pivot_modalidad = crear_pivot_seguro(
        df_detalle,
        columna_categoria="modalidad",
        prefijo="MOD"
    )
    if not pivot_modalidad.empty:
        tablas_pivot.append(pivot_modalidad)

# =========================
# 3. Super merge seguro
# =========================

df_sidpol_super_merge = df_base_distrito_mes.copy()

for tabla in tablas_pivot:
    df_sidpol_super_merge = df_sidpol_super_merge.merge(
        tabla,
        on=llaves_base,
        how="outer"
    )

# Rellenar nulos numéricos con 0
columnas_no_numericas = llaves_base + ["dist_emergencia"]

columnas_metricas = [
    c for c in df_sidpol_super_merge.columns
    if c not in columnas_no_numericas
]

df_sidpol_super_merge[columnas_metricas] = (
    df_sidpol_super_merge[columnas_metricas]
    .fillna(0)
    .astype(int)
)

# Si dist_emergencia queda nulo, dejarlo como 0
df_sidpol_super_merge["dist_emergencia"] = (
    df_sidpol_super_merge["dist_emergencia"]
    .fillna(0)
    .astype(int)
)

# Ordenar
df_sidpol_super_merge = df_sidpol_super_merge.sort_values(
    ["anio", "mes", "departamento", "provincia", "distrito"]
).reset_index(drop=True)

ruta_super_merge = carpeta_salida / "sidpol_super_merge_distrito_mes.csv"

df_sidpol_super_merge.to_csv(
    ruta_super_merge,
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Celda 5: QA del resultado final

qa_final = {}

qa_final["filas_sidpol_largo"] = len(df_sidpol_largo)
qa_final["columnas_sidpol_largo"] = len(df_sidpol_largo.columns)

qa_final["filas_super_merge"] = len(df_sidpol_super_merge)
qa_final["columnas_super_merge"] = len(df_sidpol_super_merge.columns)

llave_super_merge = [
    "anio",
    "mes",
    "ubigeo"
]

duplicados_llave = df_sidpol_super_merge.duplicated(subset=llave_super_merge).sum()

qa_final["duplicados_en_super_merge_por_anio_mes_ubigeo"] = int(duplicados_llave)

qa_final["anio_min_super_merge"] = int(df_sidpol_super_merge["anio"].min())
qa_final["anio_max_super_merge"] = int(df_sidpol_super_merge["anio"].max())

qa_final["distritos_unicos_super_merge"] = int(df_sidpol_super_merge["ubigeo"].nunique())
qa_final["departamentos_unicos_super_merge"] = int(df_sidpol_super_merge["departamento"].nunique())

qa_final["columnas_metricas_creadas"] = len([
    c for c in df_sidpol_super_merge.columns
    if c.startswith("PT__")
    or c.startswith("PM__")
    or c.startswith("TIPO__")
    or c.startswith("SUBTIPO__")
    or c.startswith("MOD__")
])

df_qa_final = pd.DataFrame(
    list(qa_final.items()),
    columns=["metrica", "valor"]
)

ruta_qa_final = carpeta_salida / "qa_super_merge_final.csv"

df_qa_final.to_csv(
    ruta_qa_final,
    index=False,
    encoding="utf-8-sig"
)

# También guardar una muestra para revisar rápido
ruta_muestra = carpeta_salida / "muestra_super_merge_1000_filas.csv"

df_sidpol_super_merge.head(1000).to_csv(
    ruta_muestra,
    index=False,
    encoding="utf-8-sig"
)

os.startfile(carpeta_salida)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata
import re

# ============================================================
# 1. RUTA DEL ARCHIVO
# ============================================================

carpeta_data = Path(
    r"C:\Users\isancheza\OneDrive - SBS\Documentos\UPC\Data Vizualitation\7304416-base-de-datos-sidpol-a-setiembre-2025(2) (1)\Data"
)

archivo_excel = None

for archivo in carpeta_data.glob("*.xlsx"):
    if "SIDPOL" in archivo.name.upper():
        archivo_excel = archivo
        break

if archivo_excel is None:
    raise FileNotFoundError("No se encontró el archivo Excel de SIDPOL.")

ruta_salida = carpeta_data / "sidpol_consolidado_final_limpio.csv"

# ============================================================
# 2. FUNCIONES DE LIMPIEZA
# ============================================================

def quitar_tildes(valor):
    if pd.isna(valor):
        return np.nan
    valor = str(valor)
    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(c for c in valor if not unicodedata.combining(c))
    return valor


def limpiar_texto(valor):
    if pd.isna(valor):
        return np.nan
    valor = quitar_tildes(valor)
    valor = valor.upper().strip()
    valor = re.sub(r"\s+", " ", valor)
    return valor


def limpiar_categoria(valor):
    if pd.isna(valor):
        return np.nan

    valor = limpiar_texto(valor)

    # Ejemplo: 1.Delitos -> DELITOS
    valor = re.sub(r"^\d+\.\s*", "", valor)

    # Quitar textos repetitivos sin perder la categoría
    valor = valor.replace("(DELITO)", "")
    valor = valor.strip()
    valor = re.sub(r"\s+", " ", valor)

    return valor


def limpiar_nombre_columna(columna):
    columna = quitar_tildes(columna)
    columna = columna.upper().strip()
    columna = re.sub(r"\s+", "_", columna)
    columna = columna.replace(".", "_")
    columna = re.sub(r"_+", "_", columna)
    return columna


def normalizar_ubigeo(valor):
    if pd.isna(valor):
        return np.nan

    try:
        valor = str(int(float(valor)))
    except:
        valor = str(valor).strip()

    valor = re.sub(r"\D", "", valor)

    if valor == "":
        return np.nan

    return valor.zfill(6)


def crear_fecha(anio, mes):
    anio = anio.astype("Int64").astype(str)
    mes = mes.astype("Int64").astype(str).str.zfill(2)
    return pd.to_datetime(anio + "-" + mes + "-01", errors="coerce")


def leer_hoja(nombre_hoja):
    df = pd.read_excel(archivo_excel, sheet_name=nombre_hoja)
    df.columns = [limpiar_nombre_columna(c) for c in df.columns]

    mapa = {
        "ANIO": "anio",
        "MES": "mes",
        "UBIGEO_HECHO": "ubigeo",
        "DPTO_HECHO_NEW": "departamento",
        "PROV_HECHO": "provincia",
        "DIST_HECHO": "distrito",
        "ES_DELITO_X": "es_delito",
        "PRINCIPALES_TIPOS": "principales_tipos",
        "PMODALIDADES": "p_modalidad",
        "P_MODALIDADES": "p_modalidad",
        "TIPO": "tipo",
        "SUB_TIPO": "sub_tipo",
        "MODALIDAD": "modalidad",
        "DIST_EMERGENCIA": "dist_emergencia",
        "N_DIST_ID_DGC": "cantidad"
    }

    df = df.rename(columns=mapa)

    columnas_necesarias = [
        "anio",
        "mes",
        "ubigeo",
        "departamento",
        "provincia",
        "distrito",
        "es_delito",
        "principales_tipos",
        "p_modalidad",
        "tipo",
        "sub_tipo",
        "modalidad",
        "dist_emergencia",
        "cantidad"
    ]

    for col in columnas_necesarias:
        if col not in df.columns:
            df[col] = np.nan

    df["anio"] = pd.to_numeric(df["anio"], errors="coerce").astype("Int64")
    df["mes"] = pd.to_numeric(df["mes"], errors="coerce").astype("Int64")
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0).astype(int)

    df["ubigeo"] = df["ubigeo"].apply(normalizar_ubigeo)

    for col in ["departamento", "provincia", "distrito"]:
        df[col] = df[col].apply(limpiar_texto)

    for col in ["es_delito", "principales_tipos", "p_modalidad", "tipo", "sub_tipo", "modalidad"]:
        df[col] = df[col].apply(limpiar_categoria)

    df["fecha"] = crear_fecha(df["anio"], df["mes"])
    df["anio_mes"] = df["fecha"].dt.strftime("%Y-%m")

    return df

# ============================================================
# 3. LEER HOJAS
# ============================================================

excel = pd.ExcelFile(archivo_excel)

df_temp2 = leer_hoja("Temp2")      # Total departamental por clasificación general
df_temp3 = leer_hoja("Temp3")      # Total departamental por principales tipos
df_temp4 = leer_hoja("Temp4")      # Total departamental por principales modalidades
df_temp5 = leer_hoja("Temp5")      # Distrito por principales tipos
df_temp52 = leer_hoja("Temp5.2")   # Distrito por principales modalidades
df_temp6 = leer_hoja("Temp6")      # Distrito detalle 2024
df_temp7 = leer_hoja("Temp7")      # Distrito detalle 2025

# ============================================================
# 4. CREAR CATÁLOGO GEOGRÁFICO DESDE HOJAS CON UBIGEO
# ============================================================

geo_base = pd.concat(
    [
        df_temp5[["ubigeo", "departamento", "provincia", "distrito"]],
        df_temp52[["ubigeo", "departamento", "provincia", "distrito"]],
        df_temp6[["ubigeo", "departamento", "provincia", "distrito"]],
        df_temp7[["ubigeo", "departamento", "provincia", "distrito"]]
    ],
    ignore_index=True
)

geo_base = geo_base.dropna(subset=["ubigeo", "departamento", "provincia", "distrito"])
geo_base = geo_base.drop_duplicates()

geo_base = (
    geo_base
    .groupby("ubigeo", as_index=False)
    .agg({
        "departamento": lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0],
        "provincia": lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0],
        "distrito": lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]
    })
)

dict_departamento = geo_base.set_index("ubigeo")["departamento"].to_dict()
dict_provincia = geo_base.set_index("ubigeo")["provincia"].to_dict()
dict_distrito = geo_base.set_index("ubigeo")["distrito"].to_dict()

# ============================================================
# 5. FUNCIÓN PARA CORREGIR GEOGRAFÍA DISTRITAL
# ============================================================

def corregir_geografia_distrital(df):
    df = df.copy()

    df = df.dropna(subset=["ubigeo"])

    df["departamento"] = df["ubigeo"].map(dict_departamento).combine_first(df["departamento"])
    df["provincia"] = df["ubigeo"].map(dict_provincia).combine_first(df["provincia"])
    df["distrito"] = df["ubigeo"].map(dict_distrito).combine_first(df["distrito"])

    df = df.dropna(subset=["departamento", "provincia", "distrito"])

    return df

# ============================================================
# 6. CREAR FILAS DISTRITALES REALES
# ============================================================

# Temp5: principales tipos por distrito
distrito_tipos = corregir_geografia_distrital(df_temp5)

distrito_tipos["tipo_analisis"] = "PRINCIPALES_TIPOS"
distrito_tipos["categoria"] = distrito_tipos["principales_tipos"]
distrito_tipos["tipo"] = distrito_tipos["principales_tipos"]
distrito_tipos["sub_tipo"] = ""
distrito_tipos["modalidad"] = ""

# Temp5.2: principales modalidades por distrito
distrito_modalidades = corregir_geografia_distrital(df_temp52)

distrito_modalidades["tipo_analisis"] = "PRINCIPALES_MODALIDADES"
distrito_modalidades["categoria"] = distrito_modalidades["p_modalidad"]
distrito_modalidades["tipo"] = ""
distrito_modalidades["sub_tipo"] = ""
distrito_modalidades["modalidad"] = distrito_modalidades["p_modalidad"]

# Temp6 y Temp7: detalle por distrito
distrito_detalle = pd.concat([df_temp6, df_temp7], ignore_index=True)
distrito_detalle = corregir_geografia_distrital(distrito_detalle)

distrito_detalle["tipo_analisis"] = "DETALLE_MODALIDAD"
distrito_detalle["categoria"] = distrito_detalle["modalidad"]

# ============================================================
# 7. UNIR FILAS DISTRITALES
# ============================================================

columnas_finales = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito",
    "dist_emergencia",
    "tipo_analisis",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",
    "cantidad"
]

df_distrital = pd.concat(
    [
        distrito_tipos[columnas_finales],
        distrito_modalidades[columnas_finales],
        distrito_detalle[columnas_finales]
    ],
    ignore_index=True
)

# Limpiar emergencia
df_distrital["dist_emergencia"] = df_distrital["dist_emergencia"].replace({
    1: "SI",
    0: "NO",
    "1": "SI",
    "0": "NO"
})

df_distrital["dist_emergencia"] = df_distrital["dist_emergencia"].fillna("")

# Quitar categorías vacías
df_distrital = df_distrital.dropna(subset=["categoria"])

# ============================================================
# 8. CREAR TOTALES DEPARTAMENTALES PARA MERGE DE CONTEXTO
# ============================================================

# 8.1 Cantidad general de DELITOS por departamento, año y mes
dept_delitos = df_temp2.copy()
dept_delitos = dept_delitos[dept_delitos["es_delito"] == "DELITOS"]

dept_delitos = (
    dept_delitos
    .groupby(["anio", "mes", "departamento"], dropna=False, as_index=False)["cantidad"]
    .max()
    .rename(columns={"cantidad": "cantidad_departamento_delitos"})
)

# 8.2 Cantidad departamental de la misma categoría para principales tipos
dept_tipos = df_temp3.copy()
dept_tipos["tipo_analisis"] = "PRINCIPALES_TIPOS"
dept_tipos["categoria"] = dept_tipos["principales_tipos"]

dept_tipos = (
    dept_tipos
    .dropna(subset=["categoria"])
    .groupby(["anio", "mes", "departamento", "tipo_analisis", "categoria"], dropna=False, as_index=False)["cantidad"]
    .max()
    .rename(columns={"cantidad": "cantidad_departamento_misma_categoria"})
)

# 8.3 Cantidad departamental de la misma categoría para principales modalidades
dept_modalidades = df_temp4.copy()
dept_modalidades["tipo_analisis"] = "PRINCIPALES_MODALIDADES"
dept_modalidades["categoria"] = dept_modalidades["p_modalidad"]

dept_modalidades = (
    dept_modalidades
    .dropna(subset=["categoria"])
    .groupby(["anio", "mes", "departamento", "tipo_analisis", "categoria"], dropna=False, as_index=False)["cantidad"]
    .max()
    .rename(columns={"cantidad": "cantidad_departamento_misma_categoria"})
)

dept_categoria = pd.concat([dept_tipos, dept_modalidades], ignore_index=True)

# ============================================================
# 9. HACER MERGE DE CONTEXTO CONTRA FILAS DISTRITALES
# ============================================================

df_final = df_distrital.merge(
    dept_delitos,
    on=["anio", "mes", "departamento"],
    how="left"
)

df_final = df_final.merge(
    dept_categoria,
    on=["anio", "mes", "departamento", "tipo_analisis", "categoria"],
    how="left"
)

# ============================================================
# 10. CREAR INDICADORES ÚTILES
# ============================================================

df_final["participacion_en_delitos_departamento_pct"] = np.where(
    df_final["cantidad_departamento_delitos"].notna() & (df_final["cantidad_departamento_delitos"] > 0),
    round((df_final["cantidad"] / df_final["cantidad_departamento_delitos"]) * 100, 4),
    np.nan
)

df_final["participacion_misma_categoria_departamento_pct"] = np.where(
    df_final["cantidad_departamento_misma_categoria"].notna() & (df_final["cantidad_departamento_misma_categoria"] > 0),
    round((df_final["cantidad"] / df_final["cantidad_departamento_misma_categoria"]) * 100, 4),
    np.nan
)

# ============================================================
# 11. ELIMINAR DUPLICADOS SIN SUMAR
# Si se repite la misma fila lógica, queda la mayor cantidad
# ============================================================

llave = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito",
    "dist_emergencia",
    "tipo_analisis",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria",
    "participacion_en_delitos_departamento_pct",
    "participacion_misma_categoria_departamento_pct"
]

df_final = (
    df_final
    .groupby(llave, dropna=False, as_index=False)["cantidad"]
    .max()
)

# ============================================================
# 12. ORDEN FINAL
# ============================================================

columnas_ordenadas = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito",
    "dist_emergencia",
    "tipo_analisis",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",
    "cantidad",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria",
    "participacion_en_delitos_departamento_pct",
    "participacion_misma_categoria_departamento_pct"
]

df_final = df_final[columnas_ordenadas]

df_final = df_final.sort_values(
    by=[
        "anio",
        "mes",
        "departamento",
        "provincia",
        "distrito",
        "tipo_analisis",
        "categoria"
    ]
).reset_index(drop=True)

# ============================================================
# 13. EXPORTAR UN SOLO CSV
# ============================================================

df_final.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

ruta_salida

In [ ]:
# Celda 3: Revisar estructura del nuevo dataset consolidado y exportar resumen en CSV

import pandas as pd
from pathlib import Path
import os

# ============================================================
# 1. RUTA DEL DATASET CONSOLIDADO
# ============================================================

carpeta_data = Path(
    r"C:\Users\isancheza\OneDrive - SBS\Documentos\UPC\Data Vizualitation\7304416-base-de-datos-sidpol-a-setiembre-2025(2) (1)\Data"
)

archivo_consolidado = carpeta_data / "sidpol_consolidado_final_limpio.csv"

if not archivo_consolidado.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {archivo_consolidado}")

# ============================================================
# 2. LEER DATASET CONSOLIDADO
# ============================================================

df_sidpol_final = pd.read_csv(
    archivo_consolidado,
    encoding="utf-8-sig"
)

# ============================================================
# 3. CREAR RESUMEN DE COLUMNAS
# ============================================================

resumen_columnas = []

for orden, columna in enumerate(df_sidpol_final.columns, start=1):
    serie = df_sidpol_final[columna]
    
    ejemplos = (
        serie
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(5)
        .tolist()
    )
    
    resumen_columnas.append({
        "orden_columna": orden,
        "nombre_columna": columna,
        "tipo_dato": str(serie.dtype),
        "filas_totales": len(df_sidpol_final),
        "valores_nulos": int(serie.isna().sum()),
        "porcentaje_nulos": round(serie.isna().mean() * 100, 2),
        "valores_unicos": int(serie.nunique(dropna=True)),
        "ejemplos_valores": " | ".join(ejemplos)
    })

df_resumen_columnas = pd.DataFrame(resumen_columnas)

# ============================================================
# 4. CREAR RESUMEN GENERAL DEL DATASET
# ============================================================

resumen_general = {
    "archivo": archivo_consolidado.name,
    "filas_totales": len(df_sidpol_final),
    "columnas_totales": len(df_sidpol_final.columns),
    "anio_min": df_sidpol_final["anio"].min() if "anio" in df_sidpol_final.columns else None,
    "anio_max": df_sidpol_final["anio"].max() if "anio" in df_sidpol_final.columns else None,
    "mes_min": df_sidpol_final["mes"].min() if "mes" in df_sidpol_final.columns else None,
    "mes_max": df_sidpol_final["mes"].max() if "mes" in df_sidpol_final.columns else None,
    "departamentos_unicos": df_sidpol_final["departamento"].nunique() if "departamento" in df_sidpol_final.columns else None,
    "provincias_unicas": df_sidpol_final["provincia"].nunique() if "provincia" in df_sidpol_final.columns else None,
    "distritos_unicos": df_sidpol_final["distrito"].nunique() if "distrito" in df_sidpol_final.columns else None,
    "ubigeos_unicos": df_sidpol_final["ubigeo"].nunique() if "ubigeo" in df_sidpol_final.columns else None,
    "tipos_analisis_unicos": df_sidpol_final["tipo_analisis"].nunique() if "tipo_analisis" in df_sidpol_final.columns else None,
    "categorias_unicas": df_sidpol_final["categoria"].nunique() if "categoria" in df_sidpol_final.columns else None,
    "cantidad_total": df_sidpol_final["cantidad"].sum() if "cantidad" in df_sidpol_final.columns else None
}

df_resumen_general = pd.DataFrame(
    list(resumen_general.items()),
    columns=["metrica", "valor"]
)

# ============================================================
# 5. RESUMEN POR TIPO DE ANÁLISIS
# ============================================================

if "tipo_analisis" in df_sidpol_final.columns:
    df_resumen_tipo_analisis = (
        df_sidpol_final
        .groupby("tipo_analisis", dropna=False)
        .agg(
            filas=("tipo_analisis", "size"),
            categorias_unicas=("categoria", "nunique"),
            departamentos_unicos=("departamento", "nunique"),
            provincias_unicas=("provincia", "nunique"),
            distritos_unicos=("distrito", "nunique"),
            ubigeos_unicos=("ubigeo", "nunique"),
            cantidad_total=("cantidad", "sum")
        )
        .reset_index()
    )
else:
    df_resumen_tipo_analisis = pd.DataFrame()

# ============================================================
# 6. GUARDAR CSV DE REVISIÓN
# ============================================================

carpeta_revision = carpeta_data / "revision_dataset_final"
carpeta_revision.mkdir(exist_ok=True)

ruta_resumen_columnas = carpeta_revision / "resumen_columnas_dataset_final.csv"
ruta_resumen_general = carpeta_revision / "resumen_general_dataset_final.csv"
ruta_resumen_tipo_analisis = carpeta_revision / "resumen_tipo_analisis_dataset_final.csv"
ruta_muestra = carpeta_revision / "muestra_1000_filas_dataset_final.csv"

df_resumen_columnas.to_csv(
    ruta_resumen_columnas,
    index=False,
    encoding="utf-8-sig"
)

df_resumen_general.to_csv(
    ruta_resumen_general,
    index=False,
    encoding="utf-8-sig"
)

df_resumen_tipo_analisis.to_csv(
    ruta_resumen_tipo_analisis,
    index=False,
    encoding="utf-8-sig"
)

df_sidpol_final.head(1000).to_csv(
    ruta_muestra,
    index=False,
    encoding="utf-8-sig"
)

# Abrir carpeta donde se guardaron los CSV
os.startfile(carpeta_revision)

In [ ]:
# Celda 3: Revisar dataset consolidado y exportar TODO en un solo CSV

import pandas as pd
from pathlib import Path
import os

# ============================================================
# 1. RUTA DEL DATASET CONSOLIDADO
# ============================================================

carpeta_data = Path(
    r"C:\Users\isancheza\OneDrive - SBS\Documentos\UPC\Data Vizualitation\7304416-base-de-datos-sidpol-a-setiembre-2025(2) (1)\Data"
)

archivo_consolidado = carpeta_data / "sidpol_consolidado_final_limpio.csv"

if not archivo_consolidado.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {archivo_consolidado}")

# ============================================================
# 2. LEER DATASET
# ============================================================

df = pd.read_csv(
    archivo_consolidado,
    encoding="utf-8-sig"
)

# ============================================================
# 3. RESUMEN GENERAL
# ============================================================

resumen_general = [
    {"seccion": "RESUMEN_GENERAL", "campo": "archivo", "valor": archivo_consolidado.name},
    {"seccion": "RESUMEN_GENERAL", "campo": "filas_totales", "valor": len(df)},
    {"seccion": "RESUMEN_GENERAL", "campo": "columnas_totales", "valor": len(df.columns)},
]

for col in ["anio", "mes", "departamento", "provincia", "distrito", "ubigeo", "tipo_analisis", "categoria"]:
    if col in df.columns:
        resumen_general.append({
            "seccion": "RESUMEN_GENERAL",
            "campo": f"{col}_unicos",
            "valor": df[col].nunique(dropna=True)
        })

if "anio" in df.columns:
    resumen_general.append({"seccion": "RESUMEN_GENERAL", "campo": "anio_min", "valor": df["anio"].min()})
    resumen_general.append({"seccion": "RESUMEN_GENERAL", "campo": "anio_max", "valor": df["anio"].max()})

if "mes" in df.columns:
    resumen_general.append({"seccion": "RESUMEN_GENERAL", "campo": "mes_min", "valor": df["mes"].min()})
    resumen_general.append({"seccion": "RESUMEN_GENERAL", "campo": "mes_max", "valor": df["mes"].max()})

if "cantidad" in df.columns:
    resumen_general.append({"seccion": "RESUMEN_GENERAL", "campo": "cantidad_total", "valor": df["cantidad"].sum()})

df_resumen_general = pd.DataFrame(resumen_general)

# ============================================================
# 4. RESUMEN DE COLUMNAS
# ============================================================

resumen_columnas = []

for orden, columna in enumerate(df.columns, start=1):
    serie = df[columna]
    
    ejemplos = (
        serie
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(5)
        .tolist()
    )
    
    resumen_columnas.append({
        "seccion": "RESUMEN_COLUMNAS",
        "campo": columna,
        "valor": "",
        "orden_columna": orden,
        "tipo_dato": str(serie.dtype),
        "filas_totales": len(df),
        "valores_nulos": int(serie.isna().sum()),
        "porcentaje_nulos": round(serie.isna().mean() * 100, 2),
        "valores_unicos": int(serie.nunique(dropna=True)),
        "ejemplos_valores": " | ".join(ejemplos)
    })

df_resumen_columnas = pd.DataFrame(resumen_columnas)

# ============================================================
# 5. RESUMEN POR TIPO DE ANÁLISIS
# ============================================================

if "tipo_analisis" in df.columns:
    df_tipo_analisis = (
        df
        .groupby("tipo_analisis", dropna=False)
        .agg(
            filas=("tipo_analisis", "size"),
            categorias_unicas=("categoria", "nunique"),
            departamentos_unicos=("departamento", "nunique"),
            provincias_unicas=("provincia", "nunique"),
            distritos_unicos=("distrito", "nunique"),
            ubigeos_unicos=("ubigeo", "nunique"),
            cantidad_total=("cantidad", "sum")
        )
        .reset_index()
    )
    
    df_tipo_analisis.insert(0, "seccion", "RESUMEN_TIPO_ANALISIS")
    df_tipo_analisis = df_tipo_analisis.rename(columns={"tipo_analisis": "campo"})
else:
    df_tipo_analisis = pd.DataFrame()

# ============================================================
# 6. TOP 20 CATEGORÍAS POR CANTIDAD
# ============================================================

if "categoria" in df.columns and "cantidad" in df.columns:
    df_top_categorias = (
        df
        .groupby("categoria", dropna=False)["cantidad"]
        .sum()
        .reset_index()
        .sort_values("cantidad", ascending=False)
        .head(20)
    )
    
    df_top_categorias.insert(0, "seccion", "TOP_20_CATEGORIAS")
    df_top_categorias = df_top_categorias.rename(columns={
        "categoria": "campo",
        "cantidad": "valor"
    })
else:
    df_top_categorias = pd.DataFrame()

# ============================================================
# 7. TOP 20 DISTRITOS POR CANTIDAD
# ============================================================

if all(col in df.columns for col in ["departamento", "provincia", "distrito", "cantidad"]):
    df_top_distritos = (
        df
        .groupby(["departamento", "provincia", "distrito"], dropna=False)["cantidad"]
        .sum()
        .reset_index()
        .sort_values("cantidad", ascending=False)
        .head(20)
    )
    
    df_top_distritos.insert(0, "seccion", "TOP_20_DISTRITOS")
    df_top_distritos["campo"] = (
        df_top_distritos["departamento"].astype(str)
        + " / "
        + df_top_distritos["provincia"].astype(str)
        + " / "
        + df_top_distritos["distrito"].astype(str)
    )
    df_top_distritos = df_top_distritos.rename(columns={"cantidad": "valor"})
    df_top_distritos = df_top_distritos[["seccion", "campo", "valor"]]
else:
    df_top_distritos = pd.DataFrame()

# ============================================================
# 8. UNIFICAR TODO EN UN SOLO CSV
# ============================================================

bloques = [
    df_resumen_general,
    df_resumen_columnas,
    df_tipo_analisis,
    df_top_categorias,
    df_top_distritos
]

df_reporte_unico = pd.concat(
    bloques,
    ignore_index=True,
    sort=False
)

# ============================================================
# 9. EXPORTAR UN SOLO CSV
# ============================================================

ruta_reporte_unico = carpeta_data / "revision_dataset_final_unico.csv"

df_reporte_unico.to_csv(
    ruta_reporte_unico,
    index=False,
    encoding="utf-8-sig"
)

os.startfile(ruta_reporte_unico)

ruta_reporte_unico

In [ ]:
# Celda 4: Mejorar CSV consolidado final

import pandas as pd
import numpy as np
from pathlib import Path
import re
import os

# ============================================================
# 1. RUTAS
# ============================================================

carpeta_data = Path(
    r"C:\Users\isancheza\OneDrive - SBS\Documentos\UPC\Data Vizualitation\7304416-base-de-datos-sidpol-a-setiembre-2025(2) (1)\Data"
)

archivo_entrada = carpeta_data / "sidpol_consolidado_final_limpio.csv"
archivo_salida = carpeta_data / "sidpol_consolidado_final_mejorado.csv"

if not archivo_entrada.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {archivo_entrada}")

# ============================================================
# 2. LEER DATASET CONSOLIDADO
# ============================================================

df = pd.read_csv(
    archivo_entrada,
    encoding="utf-8-sig",
    dtype={
        "ubigeo": str
    }
)

# ============================================================
# 3. CORREGIR UBIGEO
# ============================================================

df["ubigeo"] = (
    df["ubigeo"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(6)
)

# Columna extra para Excel.
# Si abres el CSV directamente en Excel, Excel puede borrar el cero inicial.
# Esta columna ayuda a visualizarlo correctamente en Excel.
df["ubigeo_6digitos"] = df["ubigeo"]

# ============================================================
# 4. NORMALIZAR FECHA Y VARIABLES TEMPORALES
# ============================================================

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

df["anio"] = pd.to_numeric(df["anio"], errors="coerce").astype("Int64")
df["mes"] = pd.to_numeric(df["mes"], errors="coerce").astype("Int64")

df["anio_mes"] = df["fecha"].dt.strftime("%Y-%m")
df["trimestre"] = df["fecha"].dt.quarter.astype("Int64")
df["semestre"] = np.where(df["mes"] <= 6, 1, 2)

df["periodo_trimestre"] = (
    df["anio"].astype(str)
    + "-T"
    + df["trimestre"].astype(str)
)

# ============================================================
# 5. LIMPIAR TEXTOS BÁSICOS
# ============================================================

columnas_texto = [
    "departamento",
    "provincia",
    "distrito",
    "dist_emergencia",
    "tipo_analisis",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria"
]

for col in columnas_texto:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.upper()
            .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})
        )

# ============================================================
# 6. COMPLETAR DIST_EMERGENCIA POR UBIGEO
# ============================================================

# Crear mapa usando los valores conocidos por ubigeo
df_emergencia_conocida = df[
    df["dist_emergencia"].notna()
    & df["dist_emergencia"].isin(["SI", "NO"])
].copy()

mapa_emergencia = (
    df_emergencia_conocida
    .groupby("ubigeo")["dist_emergencia"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    .to_dict()
)

df["dist_emergencia"] = df["dist_emergencia"].fillna(
    df["ubigeo"].map(mapa_emergencia)
)

df["dist_emergencia"] = df["dist_emergencia"].fillna("SIN INFORMACION")

# ============================================================
# 7. CREAR JERARQUÍA ANALÍTICA LIMPIA
# ============================================================

df["categoria_nivel_1"] = ""
df["categoria_nivel_2"] = ""
df["categoria_nivel_3"] = ""

# PRINCIPALES_TIPOS:
# La categoría representa el grupo principal.
mask_tipos = df["tipo_analisis"] == "PRINCIPALES_TIPOS"

df.loc[mask_tipos, "categoria_nivel_1"] = df.loc[mask_tipos, "categoria"].fillna("")
df.loc[mask_tipos, "categoria_nivel_2"] = ""
df.loc[mask_tipos, "categoria_nivel_3"] = ""

# PRINCIPALES_MODALIDADES:
# La categoría representa modalidad resumida.
mask_modalidades = df["tipo_analisis"] == "PRINCIPALES_MODALIDADES"

df.loc[mask_modalidades, "categoria_nivel_1"] = "PRINCIPALES MODALIDADES"
df.loc[mask_modalidades, "categoria_nivel_2"] = ""
df.loc[mask_modalidades, "categoria_nivel_3"] = df.loc[mask_modalidades, "categoria"].fillna("")

# DETALLE_MODALIDAD:
# Se usa tipo, subtipo y modalidad real.
mask_detalle = df["tipo_analisis"] == "DETALLE_MODALIDAD"

df.loc[mask_detalle, "categoria_nivel_1"] = df.loc[mask_detalle, "tipo"].fillna("")
df.loc[mask_detalle, "categoria_nivel_2"] = df.loc[mask_detalle, "sub_tipo"].fillna("")
df.loc[mask_detalle, "categoria_nivel_3"] = df.loc[mask_detalle, "modalidad"].fillna("")

# Nivel de categoría
df["nivel_categoria"] = np.select(
    [
        mask_tipos,
        mask_modalidades,
        mask_detalle
    ],
    [
        "TIPO_AGREGADO",
        "MODALIDAD_AGREGADA",
        "MODALIDAD_DETALLADA"
    ],
    default="SIN_CLASIFICAR"
)

# ============================================================
# 8. BANDERAS PARA NO DUPLICAR CONTEOS EN TABLEAU
# ============================================================

# Esta bandera sirve para KPIs generales.
# Para totales generales usa solo PRINCIPALES_TIPOS.
df["es_base_kpi_general"] = np.where(
    df["tipo_analisis"] == "PRINCIPALES_TIPOS",
    "SI",
    "NO"
)

# Esta bandera sirve para análisis de modalidades resumidas.
df["es_base_modalidades"] = np.where(
    df["tipo_analisis"] == "PRINCIPALES_MODALIDADES",
    "SI",
    "NO"
)

# Esta bandera sirve para análisis penal detallado.
# Ojo: en tu data solo aparece para 2024 y 2025.
df["es_detalle_penal"] = np.where(
    df["tipo_analisis"] == "DETALLE_MODALIDAD",
    "SI",
    "NO"
)

# Advertencia metodológica
df["regla_uso_cantidad"] = np.select(
    [
        df["tipo_analisis"] == "PRINCIPALES_TIPOS",
        df["tipo_analisis"] == "PRINCIPALES_MODALIDADES",
        df["tipo_analisis"] == "DETALLE_MODALIDAD"
    ],
    [
        "USAR PARA KPI GENERAL Y COMPARACION POR TIPO",
        "USAR PARA ANALISIS DE MODALIDADES PRINCIPALES",
        "USAR SOLO PARA DETALLE 2024-2025"
    ],
    default="REVISAR"
)

# ============================================================
# 9. REFERENCIAS DEPARTAMENTALES
# ============================================================

df["cantidad_departamento_delitos"] = pd.to_numeric(
    df["cantidad_departamento_delitos"],
    errors="coerce"
)

df["cantidad_departamento_misma_categoria"] = pd.to_numeric(
    df["cantidad_departamento_misma_categoria"],
    errors="coerce"
)

df["tiene_referencia_departamental"] = np.where(
    df["cantidad_departamento_delitos"].notna(),
    "SI",
    "NO"
)

df["tiene_referencia_departamental_categoria"] = np.where(
    df["cantidad_departamento_misma_categoria"].notna(),
    "SI",
    "NO"
)

# Recalcular participaciones
df["participacion_en_delitos_departamento_pct"] = np.where(
    df["cantidad_departamento_delitos"].notna()
    & (df["cantidad_departamento_delitos"] > 0),
    round((df["cantidad"] / df["cantidad_departamento_delitos"]) * 100, 4),
    np.nan
)

df["participacion_misma_categoria_departamento_pct"] = np.where(
    df["cantidad_departamento_misma_categoria"].notna()
    & (df["cantidad_departamento_misma_categoria"] > 0),
    round((df["cantidad"] / df["cantidad_departamento_misma_categoria"]) * 100, 4),
    np.nan
)

# ============================================================
# 10. ORDENAR COLUMNAS
# ============================================================

columnas_finales = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "trimestre",
    "semestre",
    "periodo_trimestre",
    "ubigeo",
    "ubigeo_6digitos",
    "departamento",
    "provincia",
    "distrito",
    "dist_emergencia",
    "tipo_analisis",
    "nivel_categoria",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",
    "categoria_nivel_1",
    "categoria_nivel_2",
    "categoria_nivel_3",
    "cantidad",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria",
    "participacion_en_delitos_departamento_pct",
    "participacion_misma_categoria_departamento_pct",
    "es_base_kpi_general",
    "es_base_modalidades",
    "es_detalle_penal",
    "tiene_referencia_departamental",
    "tiene_referencia_departamental_categoria",
    "regla_uso_cantidad"
]

df = df[columnas_finales]

# ============================================================
# 11. ELIMINAR DUPLICADOS EXACTOS SIN SUMAR
# ============================================================

llave = [c for c in df.columns if c != "cantidad"]

df = (
    df
    .groupby(llave, dropna=False, as_index=False)["cantidad"]
    .max()
)

# ============================================================
# 12. ORDENAR FILAS
# ============================================================

df = df.sort_values(
    by=[
        "anio",
        "mes",
        "departamento",
        "provincia",
        "distrito",
        "tipo_analisis",
        "categoria"
    ]
).reset_index(drop=True)

# ============================================================
# 13. EXPORTAR CSV MEJORADO
# ============================================================

df.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

os.startfile(archivo_salida)

archivo_salida

In [ ]:
# ============================================================
# NUEVA LIMPIEZA V2 - SIDPOL
# Lee sidpol_consolidado_final_mejorado.csv
# Crea sidpol_consolidado_final_limpio_v2.csv
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import re
import os

# ============================================================
# 1. RUTA
# ============================================================

carpeta_data = Path(
    r"C:\Users\Ian\Documents\UPC\9 ciclo\Data visualitation\Data\Data"
)

archivo_entrada = carpeta_data / "sidpol_consolidado_final_mejorado.csv"
archivo_salida = carpeta_data / "sidpol_consolidado_final_limpio_v2.csv"

if not archivo_entrada.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {archivo_entrada}")

# ============================================================
# 2. LEER DATA
# ============================================================

df = pd.read_csv(
    archivo_entrada,
    encoding="utf-8-sig",
    dtype={
        "ubigeo": str,
        "ubigeo_6digitos": str
    }
)

# ============================================================
# 3. LIMPIEZA GENERAL DE COLUMNAS Y TEXTOS
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

def limpiar_texto_columna(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .replace({
            "": pd.NA,
            "NAN": pd.NA,
            "NONE": pd.NA,
            "NULL": pd.NA,
            "NO APLICA": pd.NA
        })
    )

columnas_texto = [
    "departamento",
    "provincia",
    "distrito",
    "dist_emergencia",
    "tipo_analisis",
    "nivel_categoria",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria"
]

for col in columnas_texto:
    if col in df.columns:
        df[col] = limpiar_texto_columna(df[col])

# ============================================================
# 4. CORREGIR UBIGEO
# ============================================================

if "ubigeo" in df.columns:
    df["ubigeo"] = (
        df["ubigeo"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(6)
    )

# Si existe ubigeo_6digitos pero ubigeo quedó mal, usarlo como respaldo
if "ubigeo_6digitos" in df.columns:
    df["ubigeo_6digitos"] = (
        df["ubigeo_6digitos"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(6)
    )

    df["ubigeo"] = np.where(
        df["ubigeo"].isna() | (df["ubigeo"].str.len() != 6),
        df["ubigeo_6digitos"],
        df["ubigeo"]
    )

# ============================================================
# 5. LIMPIEZA DE FECHAS Y TIEMPO
# ============================================================

df["anio"] = pd.to_numeric(df["anio"], errors="coerce").astype("Int64")
df["mes"] = pd.to_numeric(df["mes"], errors="coerce").astype("Int64")

df["fecha"] = pd.to_datetime(
    df["anio"].astype(str) + "-" + df["mes"].astype(str).str.zfill(2) + "-01",
    errors="coerce"
)

df["anio_mes"] = df["fecha"].dt.strftime("%Y-%m")
df["trimestre"] = df["fecha"].dt.quarter.astype("Int64")

# ============================================================
# 6. LIMPIEZA NUMÉRICA
# ============================================================

columnas_numericas = [
    "cantidad",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria",
    "participacion_en_delitos_departamento_pct",
    "participacion_misma_categoria_departamento_pct"
]

for col in columnas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["cantidad"] = df["cantidad"].fillna(0).astype(int)

# ============================================================
# 7. COMPLETAR DIST_EMERGENCIA POR UBIGEO
# ============================================================

if "dist_emergencia" in df.columns:
    mapa_emergencia = (
        df[df["dist_emergencia"].isin(["SI", "NO"])]
        .groupby("ubigeo")["dist_emergencia"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
        .to_dict()
    )

    df["dist_emergencia"] = df["dist_emergencia"].fillna(
        df["ubigeo"].map(mapa_emergencia)
    )

    df["dist_emergencia"] = df["dist_emergencia"].fillna("SIN INFORMACION")

# ============================================================
# 8. COMPLETAR TIPO, SUBTIPO Y MODALIDAD SEGÚN TIPO_ANALISIS
# ============================================================

# PRINCIPALES_TIPOS:
# La categoría ya es el tipo.
mask_tipos = df["tipo_analisis"].eq("PRINCIPALES_TIPOS")

df.loc[mask_tipos, "tipo"] = df.loc[mask_tipos, "tipo"].fillna(
    df.loc[mask_tipos, "categoria"]
)

df.loc[mask_tipos, "sub_tipo"] = df.loc[mask_tipos, "sub_tipo"].fillna("AGREGADO")
df.loc[mask_tipos, "modalidad"] = df.loc[mask_tipos, "modalidad"].fillna("AGREGADO")

# PRINCIPALES_MODALIDADES:
# La categoría ya es la modalidad resumida.
mask_modalidades = df["tipo_analisis"].eq("PRINCIPALES_MODALIDADES")

df.loc[mask_modalidades, "tipo"] = df.loc[mask_modalidades, "tipo"].fillna("AGREGADO")
df.loc[mask_modalidades, "sub_tipo"] = df.loc[mask_modalidades, "sub_tipo"].fillna("AGREGADO")

df.loc[mask_modalidades, "modalidad"] = df.loc[mask_modalidades, "modalidad"].fillna(
    df.loc[mask_modalidades, "categoria"]
)

# DETALLE_MODALIDAD:
# Debe tener tipo, subtipo y modalidad.
mask_detalle = df["tipo_analisis"].eq("DETALLE_MODALIDAD")

df.loc[mask_detalle, "tipo"] = df.loc[mask_detalle, "tipo"].fillna("SIN TIPO")
df.loc[mask_detalle, "sub_tipo"] = df.loc[mask_detalle, "sub_tipo"].fillna("SIN SUBTIPO")
df.loc[mask_detalle, "modalidad"] = df.loc[mask_detalle, "modalidad"].fillna(
    df.loc[mask_detalle, "categoria"]
)

# Categoría nunca debe quedar vacía
df["categoria"] = df["categoria"].fillna(df["modalidad"])
df["categoria"] = df["categoria"].fillna(df["tipo"])
df["categoria"] = df["categoria"].fillna("SIN CATEGORIA")

# ============================================================
# 9. CREAR NIVEL DE DETALLE MÁS SIMPLE
# ============================================================

df["nivel_detalle"] = np.select(
    [
        df["tipo_analisis"].eq("PRINCIPALES_TIPOS"),
        df["tipo_analisis"].eq("PRINCIPALES_MODALIDADES"),
        df["tipo_analisis"].eq("DETALLE_MODALIDAD")
    ],
    [
        "TIPO",
        "MODALIDAD_RESUMIDA",
        "MODALIDAD_DETALLADA"
    ],
    default="SIN_CLASIFICAR"
)

# ============================================================
# 10. RECALCULAR PARTICIPACIONES
# ============================================================

df["participacion_en_delitos_departamento_pct"] = np.where(
    df["cantidad_departamento_delitos"].notna()
    & (df["cantidad_departamento_delitos"] > 0),
    round((df["cantidad"] / df["cantidad_departamento_delitos"]) * 100, 4),
    np.nan
)

df["participacion_misma_categoria_departamento_pct"] = np.where(
    df["cantidad_departamento_misma_categoria"].notna()
    & (df["cantidad_departamento_misma_categoria"] > 0),
    round((df["cantidad"] / df["cantidad_departamento_misma_categoria"]) * 100, 4),
    np.nan
)

# ============================================================
# 11. SCORE DE COMPLETITUD
# Sirve para elegir la mejor fila cuando hay duplicados
# ============================================================

campos_importantes = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito",
    "dist_emergencia",
    "tipo_analisis",
    "nivel_detalle",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",
    "cantidad",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria"
]

df["score_completitud"] = df[campos_importantes].notna().sum(axis=1)

# Dar más peso a filas con más detalle penal real
df["prioridad_tipo_analisis"] = np.select(
    [
        df["tipo_analisis"].eq("DETALLE_MODALIDAD"),
        df["tipo_analisis"].eq("PRINCIPALES_MODALIDADES"),
        df["tipo_analisis"].eq("PRINCIPALES_TIPOS")
    ],
    [
        3,
        2,
        1
    ],
    default=0
)

# ============================================================
# 12. QUITAR FILAS INÚTILES
# ============================================================

df = df.dropna(subset=[
    "anio",
    "mes",
    "fecha",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito",
    "tipo_analisis",
    "categoria"
])

df = df[df["cantidad"] > 0]

# ============================================================
# 13. ELIMINAR DUPLICADOS DE MANERA INTELIGENTE
# Se conserva la fila:
# 1. más completa
# 2. con mayor prioridad de análisis
# 3. con mayor cantidad
# ============================================================

llave_dedupe = [
    "anio",
    "mes",
    "ubigeo",
    "tipo_analisis",
    "categoria"
]

df = df.sort_values(
    by=[
        "score_completitud",
        "prioridad_tipo_analisis",
        "cantidad"
    ],
    ascending=[False, False, False]
)

df = df.drop_duplicates(
    subset=llave_dedupe,
    keep="first"
)

# ============================================================
# 14. DEJAR SOLO COLUMNAS NECESARIAS
# ============================================================

columnas_finales = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "trimestre",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito",
    "dist_emergencia",
    "tipo_analisis",
    "nivel_detalle",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",
    "cantidad",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria",
    "participacion_en_delitos_departamento_pct",
    "participacion_misma_categoria_departamento_pct",
    "score_completitud"
]

df_final = df[columnas_finales].copy()

# ============================================================
# 15. ORDENAR
# ============================================================

df_final = df_final.sort_values(
    by=[
        "anio",
        "mes",
        "departamento",
        "provincia",
        "distrito",
        "tipo_analisis",
        "categoria"
    ]
).reset_index(drop=True)

# ============================================================
# 16. EXPORTAR
# ============================================================

df_final.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

os.startfile(archivo_salida)

archivo_salida

In [ ]:
# ============================================================
# CELDA NUEVA: Merge SIDPOL + Población INEI por UBIGEO y AÑO
# Entrada:
#   - sidpol_consolidado_final_limpio_v2.csv
#   - Anexo 1.xlsx
# Salida:
#   - sidpol_consolidado_final_limpio_v3_poblacion.csv
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import re
import os
import openpyxl

# ============================================================
# 1. RUTAS
# ============================================================

carpeta_data = Path(
    r"C:\Users\Ian\Documents\UPC\9 ciclo\Data visualitation\Data\Data"
)

archivo_sidpol = carpeta_data / "sidpol_consolidado_final_limpio_v2.csv"

# Buscar automáticamente el archivo Anexo 1 en xlsx
archivos_anexo = list(carpeta_data.glob("*Anexo*1*.xlsx"))

if not archivo_sidpol.exists():
    raise FileNotFoundError(f"No se encontró el archivo SIDPOL: {archivo_sidpol}")

if len(archivos_anexo) == 0:
    raise FileNotFoundError("No se encontró el archivo Anexo 1.xlsx en la carpeta indicada.")

archivo_poblacion = archivos_anexo[0]

archivo_salida = carpeta_data / "sidpol_consolidado_final_limpio_v3_poblacion.csv"
archivo_poblacion_larga = carpeta_data / "poblacion_distrital_2018_2026_larga.csv"

# ============================================================
# 2. LEER SIDPOL
# ============================================================

df_sidpol = pd.read_csv(
    archivo_sidpol,
    encoding="utf-8-sig",
    dtype={
        "ubigeo": str
    }
)

# Corregir UBIGEO SIDPOL a 6 dígitos
df_sidpol["ubigeo"] = (
    df_sidpol["ubigeo"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(6)
)

df_sidpol["anio"] = pd.to_numeric(df_sidpol["anio"], errors="coerce").astype("Int64")
df_sidpol["mes"] = pd.to_numeric(df_sidpol["mes"], errors="coerce").astype("Int64")

# ============================================================
# 3. LEER ANEXO 1 DE POBLACIÓN
# ============================================================

df_pob_raw = pd.read_excel(
    archivo_poblacion,
    dtype=str
)

# Eliminar columnas totalmente vacías
df_pob_raw = df_pob_raw.dropna(axis=1, how="all")

# Normalizar nombres de columnas
df_pob_raw.columns = [str(c).strip() for c in df_pob_raw.columns]

# ============================================================
# 4. IDENTIFICAR COLUMNAS DEL ANEXO
# ============================================================

# En el anexo normalmente:
# Columna 1 = UBIGEO
# Columna 2 = Nombre geográfico
# Columnas siguientes = años 2018, 2019, ..., 2026

cols = list(df_pob_raw.columns)

col_ubigeo = cols[0]
col_nombre = cols[1]

# Detectar columnas que son años
columnas_anio = []

for col in df_pob_raw.columns:
    col_limpia = str(col).strip()
    if re.fullmatch(r"20\d{2}", col_limpia):
        columnas_anio.append(col)

# Si pandas leyó raro las columnas, intenta detectar años en la primera fila
if len(columnas_anio) == 0:
    primera_fila = df_pob_raw.iloc[0].astype(str).str.strip().tolist()
    
    nuevas_columnas = []
    for i, valor in enumerate(primera_fila):
        if re.fullmatch(r"20\d{2}", valor):
            nuevas_columnas.append(valor)
        elif i == 0:
            nuevas_columnas.append("ubigeo")
        elif i == 1:
            nuevas_columnas.append("nombre_geografico")
        else:
            nuevas_columnas.append(f"col_{i}")
    
    df_pob_raw.columns = nuevas_columnas
    df_pob_raw = df_pob_raw.iloc[1:].reset_index(drop=True)
    
    col_ubigeo = "ubigeo"
    col_nombre = "nombre_geografico"
    columnas_anio = [c for c in df_pob_raw.columns if re.fullmatch(r"20\d{2}", str(c))]

# Renombrar primeras columnas
df_pob = df_pob_raw.rename(columns={
    col_ubigeo: "ubigeo",
    col_nombre: "nombre_geografico"
}).copy()

# ============================================================
# 5. LIMPIAR POBLACIÓN
# ============================================================

df_pob["ubigeo"] = (
    df_pob["ubigeo"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(6)
)

df_pob["nombre_geografico"] = (
    df_pob["nombre_geografico"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Quedarnos solo con columnas necesarias
df_pob = df_pob[["ubigeo", "nombre_geografico"] + columnas_anio].copy()

# Filtrar solo UBIGEO válidos de 6 dígitos
df_pob = df_pob[df_pob["ubigeo"].str.fullmatch(r"\d{6}", na=False)]

# Crear nivel geográfico según UBIGEO
df_pob["nivel_poblacion"] = np.select(
    [
        df_pob["ubigeo"].eq("000000"),
        df_pob["ubigeo"].str.endswith("0000"),
        df_pob["ubigeo"].str.endswith("00")
    ],
    [
        "PAIS",
        "DEPARTAMENTO",
        "PROVINCIA"
    ],
    default="DISTRITO"
)

# Para el merge con SIDPOL, usamos solo distritos
df_pob_distrito = df_pob[df_pob["nivel_poblacion"] == "DISTRITO"].copy()

# ============================================================
# 6. PASAR POBLACIÓN DE FORMATO ANCHO A LARGO
# ============================================================

df_pob_larga = df_pob_distrito.melt(
    id_vars=["ubigeo", "nombre_geografico", "nivel_poblacion"],
    value_vars=columnas_anio,
    var_name="anio",
    value_name="poblacion"
)

df_pob_larga["anio"] = pd.to_numeric(df_pob_larga["anio"], errors="coerce").astype("Int64")

# Limpiar población: quitar comas, espacios, etc.
df_pob_larga["poblacion"] = (
    df_pob_larga["poblacion"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(r"[^\d]", "", regex=True)
)

df_pob_larga["poblacion"] = pd.to_numeric(
    df_pob_larga["poblacion"],
    errors="coerce"
)

# Nos quedamos con años 2018 a 2025 porque SIDPOL llega a 2025
df_pob_larga = df_pob_larga[
    df_pob_larga["anio"].between(2018, 2025)
].copy()

# Eliminar duplicados en población por ubigeo + año
df_pob_larga = (
    df_pob_larga
    .sort_values(["ubigeo", "anio"])
    .drop_duplicates(subset=["ubigeo", "anio"], keep="first")
)

# Guardar tabla de población larga por si quieres revisarla
df_pob_larga.to_csv(
    archivo_poblacion_larga,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 7. HACER MERGE SIDPOL + POBLACIÓN
# ============================================================

df_final = df_sidpol.merge(
    df_pob_larga[["ubigeo", "anio", "poblacion"]],
    on=["ubigeo", "anio"],
    how="left"
)

# ============================================================
# 8. CALCULAR TASA POR 100 MIL HABITANTES
# ============================================================

df_final["tasa_denuncias_100k"] = np.where(
    df_final["poblacion"].notna() & (df_final["poblacion"] > 0),
    round((df_final["cantidad"] / df_final["poblacion"]) * 100000, 4),
    np.nan
)

# También se puede calcular tasa departamental referencial
df_final["tasa_departamento_delitos_100k_referencial"] = np.where(
    df_final["poblacion"].notna() & (df_final["poblacion"] > 0),
    round((df_final["cantidad_departamento_delitos"] / df_final["poblacion"]) * 100000, 4),
    np.nan
)

# ============================================================
# 9. REORDENAR COLUMNAS
# ============================================================

columnas_prioritarias = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "trimestre",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito",
    "poblacion",
    "tasa_denuncias_100k",
    "dist_emergencia",
    "tipo_analisis",
    "nivel_detalle",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",
    "cantidad",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria",
    "participacion_en_delitos_departamento_pct",
    "participacion_misma_categoria_departamento_pct",
    "tasa_departamento_delitos_100k_referencial",
    "score_completitud"
]

columnas_existentes = [c for c in columnas_prioritarias if c in df_final.columns]
columnas_restantes = [c for c in df_final.columns if c not in columnas_existentes]

df_final = df_final[columnas_existentes + columnas_restantes]

# ============================================================
# 10. EXPORTAR CSV FINAL
# ============================================================

df_final.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 11. QA RÁPIDO
# ============================================================

total_filas = len(df_final)
filas_sin_poblacion = df_final["poblacion"].isna().sum()
porcentaje_sin_poblacion = round((filas_sin_poblacion / total_filas) * 100, 2)

print("MERGE TERMINADO")
print(f"Archivo SIDPOL usado: {archivo_sidpol.name}")
print(f"Archivo población usado: {archivo_poblacion.name}")
print(f"Archivo final creado: {archivo_salida.name}")
print(f"Filas finales: {total_filas:,}")
print(f"Filas sin población: {filas_sin_poblacion:,}")
print(f"% sin población: {porcentaje_sin_poblacion}%")
print(f"Archivo población larga creado: {archivo_poblacion_larga.name}")

os.startfile(archivo_salida)

In [ ]:
    # ============================================================
# CELDA NUEVA: Merge SIDPOL + Efectivos policiales PNP abril 2025
# Entrada:
#   - sidpol_consolidado_final_limpio_v3_poblacion.csv
#   - Anexo 2.xlsx
# Salida:
#   - sidpol_consolidado_final_limpio_v4_policias.csv
#   - policias_distrito_ref_2025.csv
#   - qa_merge_policias_ref_2025.csv
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata
import re
import os

# ============================================================
# 1. RUTAS
# ============================================================

carpeta_data = Path(
    r"C:\Users\Ian\Documents\UPC\9 ciclo\Data visualitation\Data\Data"
)

archivo_sidpol = carpeta_data / "sidpol_consolidado_final_limpio_v3_poblacion.csv"

# Buscar Anexo 2 en xlsx, xls o csv
posibles_anexo2 = (
    list(carpeta_data.glob("*Anexo*2*.xlsx")) +
    list(carpeta_data.glob("*Anexo*2*.xls")) +
    list(carpeta_data.glob("*Anexo*2*.csv"))
)

if not archivo_sidpol.exists():
    raise FileNotFoundError(f"No se encontró el archivo SIDPOL: {archivo_sidpol}")

if len(posibles_anexo2) == 0:
    raise FileNotFoundError("No se encontró el archivo Anexo 2 en la carpeta indicada.")

archivo_policias = posibles_anexo2[0]

archivo_salida = carpeta_data / "sidpol_consolidado_final_limpio_v4_policias.csv"
archivo_policias_distrito = carpeta_data / "policias_distrito_ref_2025.csv"
archivo_qa = carpeta_data / "qa_merge_policias_ref_2025.csv"

# ============================================================
# 2. FUNCIONES
# ============================================================

def quitar_tildes(valor):
    if pd.isna(valor):
        return np.nan
    valor = str(valor)
    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(c for c in valor if not unicodedata.combining(c))
    return valor


def limpiar_texto(valor):
    if pd.isna(valor):
        return np.nan

    valor = quitar_tildes(valor)
    valor = valor.upper().strip()
    valor = re.sub(r"\s+", " ", valor)

    # Normalizaciones útiles
    reemplazos = {
        "PROV. CONST. DEL CALLAO": "CALLAO",
        "PROVINCIA CONSTITUCIONAL DEL CALLAO": "CALLAO",
        "LIMA METROPOLITANA": "LIMA METROPOLITANA",
        "LIMA REGION": "LIMA REGION",
        "SAN MARTIN": "SAN MARTIN",
    }

    valor = reemplazos.get(valor, valor)

    return valor


def limpiar_nombre_columna(col):
    col = quitar_tildes(col)
    col = col.upper().strip()
    col = col.replace(".", "")
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col


def normalizar_ubigeo(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor)
    valor = valor.replace(".0", "")
    valor = re.sub(r"\D", "", valor)

    if valor == "":
        return np.nan

    return valor.zfill(6)


def leer_archivo_policias(ruta):
    """
    Lee Anexo 2 aunque venga en xlsx, xls o csv.
    """
    if ruta.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(ruta, dtype=str)
    elif ruta.suffix.lower() == ".csv":
        return pd.read_csv(ruta, dtype=str, encoding="utf-8-sig")
    else:
        raise ValueError(f"Formato no soportado: {ruta.suffix}")


def encontrar_columna(df, posibles):
    """
    Busca una columna usando varios posibles nombres.
    """
    cols = list(df.columns)
    for p in posibles:
        if p in cols:
            return p
    return None

# ============================================================
# 3. LEER SIDPOL
# ============================================================

df_sidpol = pd.read_csv(
    archivo_sidpol,
    encoding="utf-8-sig",
    dtype={"ubigeo": str}
)

df_sidpol.columns = [limpiar_nombre_columna(c).lower() for c in df_sidpol.columns]

df_sidpol["ubigeo"] = df_sidpol["ubigeo"].apply(normalizar_ubigeo)

for col in ["departamento", "provincia", "distrito"]:
    df_sidpol[col] = df_sidpol[col].apply(limpiar_texto)

df_sidpol["anio"] = pd.to_numeric(df_sidpol["anio"], errors="coerce").astype("Int64")
df_sidpol["mes"] = pd.to_numeric(df_sidpol["mes"], errors="coerce").astype("Int64")
df_sidpol["cantidad"] = pd.to_numeric(df_sidpol["cantidad"], errors="coerce").fillna(0)

if "poblacion" in df_sidpol.columns:
    df_sidpol["poblacion"] = pd.to_numeric(df_sidpol["poblacion"], errors="coerce")

# ============================================================
# 4. CREAR CATÁLOGO GEOGRÁFICO DESDE SIDPOL
# ============================================================

geo_sidpol = (
    df_sidpol[["ubigeo", "departamento", "provincia", "distrito"]]
    .dropna(subset=["ubigeo", "departamento", "provincia", "distrito"])
    .drop_duplicates()
)

geo_sidpol = (
    geo_sidpol
    .groupby(["departamento", "provincia", "distrito"], as_index=False)
    .agg(
        ubigeo=("ubigeo", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    )
)

# ============================================================
# 5. LEER ANEXO 2 DE POLICÍAS
# ============================================================

df_pol_raw = leer_archivo_policias(archivo_policias)

# Eliminar columnas vacías
df_pol_raw = df_pol_raw.dropna(axis=1, how="all")

# Normalizar columnas
df_pol_raw.columns = [limpiar_nombre_columna(c) for c in df_pol_raw.columns]

# Detectar columnas necesarias
col_departamento = encontrar_columna(df_pol_raw, ["DEPARTAMENTO"])
col_provincia = encontrar_columna(df_pol_raw, ["PROVINCIA"])
col_distrito = encontrar_columna(df_pol_raw, ["DISTRITO"])
col_cantidad = encontrar_columna(df_pol_raw, ["CANTIDAD"])
col_sexo = encontrar_columna(df_pol_raw, ["SEXO"])
col_categoria = encontrar_columna(df_pol_raw, ["CATEGORIA_OFI_SUB", "CATEGORIA_OFICIAL_SUBOFICIAL"])
col_arm_serv = encontrar_columna(df_pol_raw, ["CATEGORIA_ARM_SERV"])
col_funcion = encontrar_columna(df_pol_raw, ["FUNCION"])

columnas_obligatorias = {
    "DEPARTAMENTO": col_departamento,
    "PROVINCIA": col_provincia,
    "DISTRITO": col_distrito,
    "CANTIDAD": col_cantidad
}

faltantes = [k for k, v in columnas_obligatorias.items() if v is None]

if len(faltantes) > 0:
    raise ValueError(f"Faltan columnas obligatorias en Anexo 2: {faltantes}")

# Renombrar a estándar
df_pol = df_pol_raw.rename(columns={
    col_departamento: "departamento",
    col_provincia: "provincia",
    col_distrito: "distrito",
    col_cantidad: "cantidad_policias_original"
}).copy()

if col_sexo:
    df_pol = df_pol.rename(columns={col_sexo: "sexo"})

if col_categoria:
    df_pol = df_pol.rename(columns={col_categoria: "categoria_ofi_sub"})

if col_arm_serv:
    df_pol = df_pol.rename(columns={col_arm_serv: "categoria_arm_serv"})

if col_funcion:
    df_pol = df_pol.rename(columns={col_funcion: "funcion"})

# Crear columnas opcionales si no existen
for col in ["sexo", "categoria_ofi_sub", "categoria_arm_serv", "funcion"]:
    if col not in df_pol.columns:
        df_pol[col] = np.nan

# Limpiar textos
for col in ["departamento", "provincia", "distrito", "sexo", "categoria_ofi_sub", "categoria_arm_serv", "funcion"]:
    df_pol[col] = df_pol[col].apply(limpiar_texto)

# Limpiar cantidad
df_pol["cantidad_policias_original"] = (
    df_pol["cantidad_policias_original"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(r"[^\d]", "", regex=True)
)

df_pol["cantidad_policias_original"] = pd.to_numeric(
    df_pol["cantidad_policias_original"],
    errors="coerce"
).fillna(0).astype(int)

# Quitar filas inválidas
df_pol = df_pol.dropna(subset=["departamento", "provincia", "distrito"])
df_pol = df_pol[df_pol["cantidad_policias_original"] > 0]

# ============================================================
# 6. AGRUPAR POLICÍAS POR DISTRITO
# ============================================================

# Total de policías por distrito
pol_total = (
    df_pol
    .groupby(["departamento", "provincia", "distrito"], as_index=False)
    .agg(
        n_policias_ref_2025=("cantidad_policias_original", "sum")
    )
)

# Por sexo
pol_sexo = (
    df_pol
    .pivot_table(
        index=["departamento", "provincia", "distrito"],
        columns="sexo",
        values="cantidad_policias_original",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

pol_sexo.columns.name = None

# Renombrar columnas de sexo
rename_sexo = {}

if "MASCULINO" in pol_sexo.columns:
    rename_sexo["MASCULINO"] = "n_policias_masculino_ref_2025"

if "FEMENINO" in pol_sexo.columns:
    rename_sexo["FEMENINO"] = "n_policias_femenino_ref_2025"

pol_sexo = pol_sexo.rename(columns=rename_sexo)

# Por categoría oficial/suboficial
pol_categoria = (
    df_pol
    .pivot_table(
        index=["departamento", "provincia", "distrito"],
        columns="categoria_ofi_sub",
        values="cantidad_policias_original",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

pol_categoria.columns.name = None

rename_cat = {}

if "OFICIAL" in pol_categoria.columns:
    rename_cat["OFICIAL"] = "n_policias_oficial_ref_2025"

if "SUB OFICIAL" in pol_categoria.columns:
    rename_cat["SUB OFICIAL"] = "n_policias_suboficial_ref_2025"

if "SUBOFICIAL" in pol_categoria.columns:
    rename_cat["SUBOFICIAL"] = "n_policias_suboficial_ref_2025"

pol_categoria = pol_categoria.rename(columns=rename_cat)

# Por función
pol_funcion = (
    df_pol
    .pivot_table(
        index=["departamento", "provincia", "distrito"],
        columns="funcion",
        values="cantidad_policias_original",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

pol_funcion.columns.name = None

rename_funcion = {}

if "OPE" in pol_funcion.columns:
    rename_funcion["OPE"] = "n_policias_operativo_ref_2025"

if "ADM" in pol_funcion.columns:
    rename_funcion["ADM"] = "n_policias_administrativo_ref_2025"

pol_funcion = pol_funcion.rename(columns=rename_funcion)

# ============================================================
# 7. UNIR RESÚMENES DE POLICÍAS
# ============================================================

df_policias_distrito = pol_total.copy()

for tabla in [pol_sexo, pol_categoria, pol_funcion]:
    df_policias_distrito = df_policias_distrito.merge(
        tabla,
        on=["departamento", "provincia", "distrito"],
        how="left"
    )

# Mantener solo columnas útiles
columnas_utiles_pol = [
    "departamento",
    "provincia",
    "distrito",
    "n_policias_ref_2025",
    "n_policias_masculino_ref_2025",
    "n_policias_femenino_ref_2025",
    "n_policias_oficial_ref_2025",
    "n_policias_suboficial_ref_2025",
    "n_policias_operativo_ref_2025",
    "n_policias_administrativo_ref_2025"
]

columnas_utiles_pol = [c for c in columnas_utiles_pol if c in df_policias_distrito.columns]

df_policias_distrito = df_policias_distrito[columnas_utiles_pol].copy()

# Rellenar valores de conteo con 0
cols_conteo_pol = [c for c in df_policias_distrito.columns if c.startswith("n_policias")]

df_policias_distrito[cols_conteo_pol] = (
    df_policias_distrito[cols_conteo_pol]
    .fillna(0)
    .astype(int)
)

# ============================================================
# 8. AGREGAR UBIGEO A POLICÍAS USANDO CATÁLOGO SIDPOL
# ============================================================

df_policias_distrito = df_policias_distrito.merge(
    geo_sidpol,
    on=["departamento", "provincia", "distrito"],
    how="left"
)

# Reordenar
cols_pol_final = [
    "ubigeo",
    "departamento",
    "provincia",
    "distrito"
] + cols_conteo_pol

df_policias_distrito = df_policias_distrito[cols_pol_final]

# Fuente
df_policias_distrito["anio_fuente_policias"] = 2025
df_policias_distrito["mes_fuente_policias"] = 4
df_policias_distrito["fuente_policias"] = "MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025"

# Guardar tabla limpia de policías por distrito
df_policias_distrito.to_csv(
    archivo_policias_distrito,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 9. MERGE CON SIDPOL
# Se hace por departamento + provincia + distrito
# porque Anexo 2 no trae UBIGEO.
# ============================================================

cols_merge_pol = [
    "departamento",
    "provincia",
    "distrito"
]

df_final = df_sidpol.merge(
    df_policias_distrito.drop(columns=["ubigeo"], errors="ignore"),
    on=cols_merge_pol,
    how="left"
)

# ============================================================
# 10. COMPLETAR POLICÍAS EN 0 SI NO HUBO MATCH
# ============================================================

cols_n_policias = [c for c in df_final.columns if c.startswith("n_policias")]

for col in cols_n_policias:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce").fillna(0).astype(int)

df_final["anio_fuente_policias"] = df_final["anio_fuente_policias"].fillna(2025).astype(int)
df_final["mes_fuente_policias"] = df_final["mes_fuente_policias"].fillna(4).astype(int)

df_final["fuente_policias"] = df_final["fuente_policias"].fillna(
    "SIN MATCH CON ANEXO 2 - EFECTIVOS PNP ABRIL 2025"
)

df_final["match_policias"] = np.where(
    df_final["n_policias_ref_2025"] > 0,
    "SI",
    "NO"
)

# ============================================================
# 11. CALCULAR INDICADORES NUEVOS
# ============================================================

if "poblacion" in df_final.columns:
    df_final["policias_por_100k_ref_2025"] = np.where(
        df_final["poblacion"].notna()
        & (df_final["poblacion"] > 0)
        & (df_final["n_policias_ref_2025"] > 0),
        round((df_final["n_policias_ref_2025"] / df_final["poblacion"]) * 100000, 4),
        np.nan
    )
else:
    df_final["policias_por_100k_ref_2025"] = np.nan

df_final["denuncias_por_policia_ref_2025"] = np.where(
    df_final["n_policias_ref_2025"] > 0,
    round(df_final["cantidad"] / df_final["n_policias_ref_2025"], 4),
    np.nan
)

df_final["pct_policias_femenino_ref_2025"] = np.where(
    ("n_policias_femenino_ref_2025" in df_final.columns)
    & (df_final["n_policias_ref_2025"] > 0),
    round((df_final.get("n_policias_femenino_ref_2025", 0) / df_final["n_policias_ref_2025"]) * 100, 4),
    np.nan
)

df_final["pct_policias_operativo_ref_2025"] = np.where(
    ("n_policias_operativo_ref_2025" in df_final.columns)
    & (df_final["n_policias_ref_2025"] > 0),
    round((df_final.get("n_policias_operativo_ref_2025", 0) / df_final["n_policias_ref_2025"]) * 100, 4),
    np.nan
)

# ============================================================
# 12. ORDENAR COLUMNAS
# ============================================================

columnas_prioritarias = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "trimestre",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito",
    "poblacion",
    "tasa_denuncias_100k",
    "n_policias_ref_2025",
    "policias_por_100k_ref_2025",
    "denuncias_por_policia_ref_2025",
    "n_policias_masculino_ref_2025",
    "n_policias_femenino_ref_2025",
    "pct_policias_femenino_ref_2025",
    "n_policias_oficial_ref_2025",
    "n_policias_suboficial_ref_2025",
    "n_policias_operativo_ref_2025",
    "n_policias_administrativo_ref_2025",
    "pct_policias_operativo_ref_2025",
    "match_policias",
    "anio_fuente_policias",
    "mes_fuente_policias",
    "fuente_policias",
    "dist_emergencia",
    "tipo_analisis",
    "nivel_detalle",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",
    "cantidad",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria",
    "participacion_en_delitos_departamento_pct",
    "participacion_misma_categoria_departamento_pct",
    "tasa_departamento_delitos_100k_referencial",
    "score_completitud"
]

columnas_existentes = [c for c in columnas_prioritarias if c in df_final.columns]
columnas_restantes = [c for c in df_final.columns if c not in columnas_existentes]

df_final = df_final[columnas_existentes + columnas_restantes]

# ============================================================
# 13. EXPORTAR DATA FINAL
# ============================================================

df_final.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 14. QA DEL MERGE
# ============================================================

qa = []

qa.append({
    "metrica": "archivo_sidpol_usado",
    "valor": archivo_sidpol.name
})

qa.append({
    "metrica": "archivo_policias_usado",
    "valor": archivo_policias.name
})

qa.append({
    "metrica": "filas_sidpol_final",
    "valor": len(df_final)
})

qa.append({
    "metrica": "distritos_sidpol_unicos",
    "valor": df_sidpol[["departamento", "provincia", "distrito"]].drop_duplicates().shape[0]
})

qa.append({
    "metrica": "distritos_policias_unicos_anexo2",
    "valor": df_policias_distrito[["departamento", "provincia", "distrito"]].drop_duplicates().shape[0]
})

qa.append({
    "metrica": "distritos_policias_con_ubigeo",
    "valor": df_policias_distrito["ubigeo"].notna().sum()
})

qa.append({
    "metrica": "distritos_policias_sin_ubigeo",
    "valor": df_policias_distrito["ubigeo"].isna().sum()
})

qa.append({
    "metrica": "filas_sidpol_con_match_policias",
    "valor": int((df_final["match_policias"] == "SI").sum())
})

qa.append({
    "metrica": "filas_sidpol_sin_match_policias",
    "valor": int((df_final["match_policias"] == "NO").sum())
})

qa.append({
    "metrica": "porcentaje_filas_con_match_policias",
    "valor": round((df_final["match_policias"] == "SI").mean() * 100, 2)
})

qa.append({
    "metrica": "total_policias_anexo2",
    "valor": int(df_pol["cantidad_policias_original"].sum())
})

df_qa = pd.DataFrame(qa)

df_qa.to_csv(
    archivo_qa,
    index=False,
    encoding="utf-8-sig"
)

print("MERGE CON POLICÍAS TERMINADO")
print(f"Archivo final creado: {archivo_salida.name}")
print(f"Tabla policías por distrito creada: {archivo_policias_distrito.name}")
print(f"QA creado: {archivo_qa.name}")
print(f"Filas finales: {len(df_final):,}")
print(f"Filas con match de policías: {(df_final['match_policias'] == 'SI').sum():,}")
print(f"Filas sin match de policías: {(df_final['match_policias'] == 'NO').sum():,}")
print(f"% match: {round((df_final['match_policias'] == 'SI').mean() * 100, 2)}%")

os.startfile(archivo_salida)

In [ ]:
# ============================================================
# CELDA NUEVA: Merge SIDPOL + Comisarías Anexo 3 por nombres geográficos
# NO usa CODIGO INEI como UBIGEO
# Usa: departamento + provincia + distrito
#
# Entrada:
#   - sidpol_consolidado_final_limpio_v4_policias.csv
#     o sidpol_consolidado_final_limpio_v3_poblacion.csv
#   - Anexo 3.xlsx
#
# Salida:
#   - sidpol_consolidado_final_limpio_v5_comisarias_2020_por_nombre.csv
#   - comisarias_distrito_ref_2020_por_nombre.csv
#   - qa_merge_comisarias_2020_por_nombre.csv
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata
import re
import os
import sys
import subprocess

# ============================================================
# 1. INSTALAR OPENPYXL SI FALTA
# ============================================================

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    import openpyxl

# ============================================================
# 2. RUTAS
# ============================================================

carpeta_data = Path(
    r"C:\Users\Ian\Documents\UPC\9 ciclo\Data visualitation\Data\Data"
)

candidatos_sidpol = [
    carpeta_data / "sidpol_consolidado_final_limpio_v4_policias.csv",
    carpeta_data / "sidpol_consolidado_final_limpio_v3_poblacion.csv",
    carpeta_data / "sidpol_consolidado_final_limpio_v2.csv"
]

archivo_sidpol = None

for archivo in candidatos_sidpol:
    if archivo.exists():
        archivo_sidpol = archivo
        break

if archivo_sidpol is None:
    raise FileNotFoundError("No se encontró ningún dataset SIDPOL limpio v2, v3 o v4.")

posibles_anexo3 = (
    list(carpeta_data.glob("*Anexo*3*.xlsx")) +
    list(carpeta_data.glob("*anexo*3*.xlsx")) +
    list(carpeta_data.glob("*Anexo*3*.xls")) +
    list(carpeta_data.glob("*anexo*3*.xls"))
)

if len(posibles_anexo3) == 0:
    raise FileNotFoundError("No se encontró Anexo 3 en Excel dentro de la carpeta.")

archivo_comisarias = posibles_anexo3[0]

archivo_salida = carpeta_data / "sidpol_consolidado_final_limpio_v5_comisarias_2020_por_nombre.csv"
archivo_comisarias_distrito = carpeta_data / "comisarias_distrito_ref_2020_por_nombre.csv"
archivo_qa = carpeta_data / "qa_merge_comisarias_2020_por_nombre.csv"
archivo_no_match = carpeta_data / "comisarias_sin_match_sidpol_2020_por_nombre.csv"

print("Archivo SIDPOL usado:", archivo_sidpol.name)
print("Archivo Anexo 3 usado:", archivo_comisarias.name)

# ============================================================
# 3. FUNCIONES
# ============================================================

def quitar_tildes(valor):
    if pd.isna(valor):
        return np.nan
    
    valor = str(valor)
    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(c for c in valor if not unicodedata.combining(c))
    return valor


def limpiar_texto(valor):
    if pd.isna(valor):
        return np.nan
    
    valor = quitar_tildes(valor)
    valor = valor.upper().strip()
    valor = re.sub(r"\s+", " ", valor)
    
    # Normalizaciones comunes
    reemplazos = {
        "PROV. CONST. DEL CALLAO": "CALLAO",
        "PROVINCIA CONSTITUCIONAL DEL CALLAO": "CALLAO",
        "CALLAO": "CALLAO",
        "LIMA REGION": "LIMA REGION",
        "LIMA PROVINCIAS": "LIMA REGION",
        "LIMA METROPOLITANA": "LIMA METROPOLITANA",
        "SAN MARTIN": "SAN MARTIN",
    }
    
    return reemplazos.get(valor, valor)


def limpiar_nombre_columna(col):
    col = quitar_tildes(col)
    col = col.upper().strip()
    col = col.replace("*", "")
    col = col.replace(".", "")
    col = col.replace("/", "_")
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"[^A-Z0-9_]", "", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")


def normalizar_ubigeo(valor):
    if pd.isna(valor):
        return np.nan
    
    valor = str(valor)
    valor = valor.replace(".0", "")
    valor = re.sub(r"\D", "", valor)
    
    if valor == "":
        return np.nan
    
    return valor.zfill(6)


def encontrar_columna(df, posibles):
    for posible in posibles:
        if posible in df.columns:
            return posible
    return None


def partir_gps(valor):
    if pd.isna(valor):
        return pd.Series([np.nan, np.nan])
    
    texto = str(valor).strip().replace(" ", "")
    partes = texto.split(",")
    
    if len(partes) != 2:
        return pd.Series([np.nan, np.nan])
    
    lat = pd.to_numeric(partes[0], errors="coerce")
    lon = pd.to_numeric(partes[1], errors="coerce")
    
    return pd.Series([lat, lon])


def crear_clave_geo(df):
    df = df.copy()
    
    df["departamento_key"] = df["departamento"].apply(limpiar_texto)
    df["provincia_key"] = df["provincia"].apply(limpiar_texto)
    df["distrito_key"] = df["distrito"].apply(limpiar_texto)
    
    # Ajuste especial:
    # En algunas fuentes Lima Metropolitana aparece como departamento,
    # pero en otras aparece como Lima.
    df["departamento_key"] = df["departamento_key"].replace({
        "LIMA": "LIMA METROPOLITANA"
    })
    
    return df

# ============================================================
# 4. LEER SIDPOL
# ============================================================

df_sidpol = pd.read_csv(
    archivo_sidpol,
    encoding="utf-8-sig",
    dtype={"ubigeo": str}
)

df_sidpol.columns = [limpiar_nombre_columna(c).lower() for c in df_sidpol.columns]

df_sidpol["ubigeo"] = df_sidpol["ubigeo"].apply(normalizar_ubigeo)

for col in ["departamento", "provincia", "distrito"]:
    df_sidpol[col] = df_sidpol[col].apply(limpiar_texto)

df_sidpol["cantidad"] = pd.to_numeric(df_sidpol["cantidad"], errors="coerce").fillna(0)

if "poblacion" in df_sidpol.columns:
    df_sidpol["poblacion"] = pd.to_numeric(df_sidpol["poblacion"], errors="coerce")

df_sidpol = crear_clave_geo(df_sidpol)

# ============================================================
# 5. CREAR CATÁLOGO GEOGRÁFICO DE SIDPOL
# ============================================================

geo_sidpol = (
    df_sidpol[
        [
            "ubigeo",
            "departamento",
            "provincia",
            "distrito",
            "departamento_key",
            "provincia_key",
            "distrito_key"
        ]
    ]
    .dropna(subset=["ubigeo", "departamento_key", "provincia_key", "distrito_key"])
    .drop_duplicates()
)

# Si hubiera más de un ubigeo para una misma combinación textual,
# nos quedamos con la moda.
geo_sidpol = (
    geo_sidpol
    .groupby(["departamento_key", "provincia_key", "distrito_key"], as_index=False)
    .agg(
        ubigeo=("ubigeo", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
        departamento=("departamento", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
        provincia=("provincia", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
        distrito=("distrito", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    )
)

# ============================================================
# 6. LEER ANEXO 3
# ============================================================

excel_com = pd.ExcelFile(archivo_comisarias)

hoja_elegida = None

for hoja in excel_com.sheet_names:
    temp = pd.read_excel(archivo_comisarias, sheet_name=hoja, dtype=str, nrows=10)
    temp = temp.dropna(axis=1, how="all")
    temp.columns = [limpiar_nombre_columna(c) for c in temp.columns]
    
    if any(c in temp.columns for c in ["NOMBREDD", "NOMBREPP", "NOMBREDI", "COMISARIA"]):
        hoja_elegida = hoja
        break

if hoja_elegida is None:
    hoja_elegida = excel_com.sheet_names[0]

df_com_raw = pd.read_excel(
    archivo_comisarias,
    sheet_name=hoja_elegida,
    dtype=str
)

df_com_raw = df_com_raw.dropna(axis=1, how="all")
df_com_raw.columns = [limpiar_nombre_columna(c) for c in df_com_raw.columns]

print("Hoja de Anexo 3 usada:", hoja_elegida)
print("Columnas detectadas:", list(df_com_raw.columns))

# ============================================================
# 7. DETECTAR COLUMNAS DEL ANEXO 3
# ============================================================

col_codigo_inei = encontrar_columna(df_com_raw, [
    "CODIGO_INEI",
    "COD_INEI"
])

col_codigo_cpnp = encontrar_columna(df_com_raw, [
    "CODIGO_CPNP",
    "COD_CPNP"
])

col_departamento = encontrar_columna(df_com_raw, [
    "NOMBREDD",
    "NOMBDP",
    "NOMBDEP",
    "DEPARTAMENTO"
])

col_provincia = encontrar_columna(df_com_raw, [
    "NOMBREPP",
    "NOMBPROV",
    "PROVINCIA"
])

col_distrito = encontrar_columna(df_com_raw, [
    "NOMBREDI",
    "NOMBDIST",
    "DISTRITO"
])

col_comisaria = encontrar_columna(df_com_raw, [
    "COMISARIA",
    "COMISARIA_1",
    "NOMBRE_COMISARIA"
])

col_tipo = encontrar_columna(df_com_raw, [
    "TIPO",
    "TIPO_COMISARIA"
])

col_rural = encontrar_columna(df_com_raw, ["RURAL"])
col_sectorial = encontrar_columna(df_com_raw, ["SECTORIAL"])
col_zonal = encontrar_columna(df_com_raw, ["ZONAL"])
col_gps = encontrar_columna(df_com_raw, ["GPS"])

obligatorias = {
    "departamento": col_departamento,
    "provincia": col_provincia,
    "distrito": col_distrito,
    "comisaria": col_comisaria
}

faltantes = [k for k, v in obligatorias.items() if v is None]

if len(faltantes) > 0:
    raise ValueError(
        f"Faltan columnas obligatorias en Anexo 3: {faltantes}. "
        f"Columnas detectadas: {list(df_com_raw.columns)}"
    )

# ============================================================
# 8. LIMPIAR ANEXO 3
# ============================================================

renombrar = {
    col_departamento: "departamento",
    col_provincia: "provincia",
    col_distrito: "distrito",
    col_comisaria: "comisaria"
}

if col_codigo_inei:
    renombrar[col_codigo_inei] = "codigo_inei_anexo3"

if col_codigo_cpnp:
    renombrar[col_codigo_cpnp] = "codigo_cpnp"

if col_tipo:
    renombrar[col_tipo] = "tipo_comisaria"

if col_rural:
    renombrar[col_rural] = "rural"

if col_sectorial:
    renombrar[col_sectorial] = "sectorial"

if col_zonal:
    renombrar[col_zonal] = "zonal"

if col_gps:
    renombrar[col_gps] = "gps"

df_com = df_com_raw.rename(columns=renombrar).copy()

for col in ["codigo_inei_anexo3", "codigo_cpnp", "tipo_comisaria", "rural", "sectorial", "zonal", "gps"]:
    if col not in df_com.columns:
        df_com[col] = np.nan

for col in ["departamento", "provincia", "distrito", "comisaria", "tipo_comisaria", "rural", "sectorial", "zonal"]:
    df_com[col] = df_com[col].apply(limpiar_texto)

df_com["codigo_inei_anexo3"] = (
    df_com["codigo_inei_anexo3"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .replace({"": np.nan, "NAN": np.nan})
)

df_com["codigo_cpnp"] = (
    df_com["codigo_cpnp"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .replace({"": np.nan, "NAN": np.nan})
)

df_com = df_com.dropna(subset=["departamento", "provincia", "distrito", "comisaria"])

df_com[["latitud_comisaria", "longitud_comisaria"]] = df_com["gps"].apply(partir_gps)

# Clave geográfica textual
df_com = crear_clave_geo(df_com)

# Quitar duplicados.
# Preferimos codigo_cpnp si existe; si no, usamos nombre + ubicación.
if df_com["codigo_cpnp"].notna().sum() > 0:
    df_com = df_com.drop_duplicates(
        subset=["departamento_key", "provincia_key", "distrito_key", "codigo_cpnp"],
        keep="first"
    )
else:
    df_com = df_com.drop_duplicates(
        subset=["departamento_key", "provincia_key", "distrito_key", "comisaria"],
        keep="first"
    )

# ============================================================
# 9. TRAER UBIGEO REAL DE SIDPOL A LAS COMISARÍAS
# ============================================================

df_com = df_com.merge(
    geo_sidpol[
        [
            "departamento_key",
            "provincia_key",
            "distrito_key",
            "ubigeo",
            "departamento",
            "provincia",
            "distrito"
        ]
    ].rename(columns={
        "departamento": "departamento_sidpol",
        "provincia": "provincia_sidpol",
        "distrito": "distrito_sidpol"
    }),
    on=["departamento_key", "provincia_key", "distrito_key"],
    how="left"
)

# ============================================================
# 10. CREAR INDICADORES DE COMISARÍAS
# ============================================================

df_com["es_rural"] = np.where(df_com["rural"].eq("X"), 1, 0)
df_com["es_sectorial"] = np.where(df_com["sectorial"].eq("X"), 1, 0)
df_com["es_zonal"] = np.where(df_com["zonal"].eq("X"), 1, 0)

df_com["tipo_comisaria"] = df_com["tipo_comisaria"].fillna("SIN TIPO")

# ============================================================
# 11. AGRUPAR COMISARÍAS POR DISTRITO
# ============================================================

df_comisarias_distrito = (
    df_com
    .dropna(subset=["ubigeo"])
    .groupby(
        [
            "ubigeo",
            "departamento_sidpol",
            "provincia_sidpol",
            "distrito_sidpol"
        ],
        as_index=False
    )
    .agg(
        n_comisarias_ref_2020=("comisaria", "nunique"),
        n_comisarias_rurales_ref_2020=("es_rural", "sum"),
        n_comisarias_sectoriales_ref_2020=("es_sectorial", "sum"),
        n_comisarias_zonales_ref_2020=("es_zonal", "sum"),
        latitud_promedio_comisaria_ref_2020=("latitud_comisaria", "mean"),
        longitud_promedio_comisaria_ref_2020=("longitud_comisaria", "mean")
    )
    .rename(columns={
        "departamento_sidpol": "departamento",
        "provincia_sidpol": "provincia",
        "distrito_sidpol": "distrito"
    })
)

# Conteo por tipo de comisaría
tipos_comisaria = (
    df_com
    .dropna(subset=["ubigeo"])
    .pivot_table(
        index=["ubigeo"],
        columns="tipo_comisaria",
        values="comisaria",
        aggfunc="nunique",
        fill_value=0
    )
    .reset_index()
)

tipos_comisaria.columns.name = None

renombrar_tipos = {}

for col in tipos_comisaria.columns:
    if col == "ubigeo":
        continue
    
    nombre_seguro = (
        str(col)
        .upper()
        .strip()
        .replace(" ", "_")
        .replace(".", "")
    )
    
    renombrar_tipos[col] = f"n_comisarias_tipo_{nombre_seguro}_ref_2020"

tipos_comisaria = tipos_comisaria.rename(columns=renombrar_tipos)

df_comisarias_distrito = df_comisarias_distrito.merge(
    tipos_comisaria,
    on="ubigeo",
    how="left"
)

df_comisarias_distrito["anio_fuente_comisarias"] = 2020
df_comisarias_distrito["fuente_comisarias"] = "ANEXO 3 - RELACION DE COMISARIAS BASICAS 2020, MERGE POR NOMBRES"

# Guardar tabla por distrito
df_comisarias_distrito.to_csv(
    archivo_comisarias_distrito,
    index=False,
    encoding="utf-8-sig"
)

# Guardar comisarías que no encontraron match con SIDPOL
df_com_sin_match = df_com[df_com["ubigeo"].isna()].copy()

df_com_sin_match.to_csv(
    archivo_no_match,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 12. MERGE CON SIDPOL POR UBIGEO REAL DE SIDPOL
# ============================================================

cols_comisarias_para_merge = [
    c for c in df_comisarias_distrito.columns
    if c not in ["departamento", "provincia", "distrito"]
]

df_final = df_sidpol.merge(
    df_comisarias_distrito[cols_comisarias_para_merge],
    on="ubigeo",
    how="left"
)

# ============================================================
# 13. COMPLETAR Y CALCULAR INDICADORES
# ============================================================

cols_n_comisarias = [
    c for c in df_final.columns
    if c.startswith("n_comisarias")
]

for col in cols_n_comisarias:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce").fillna(0).astype(int)

df_final["match_comisarias_ref_2020"] = np.where(
    df_final["n_comisarias_ref_2020"] > 0,
    "SI",
    "NO"
)

df_final["anio_fuente_comisarias"] = df_final["anio_fuente_comisarias"].fillna(2020).astype(int)

df_final["fuente_comisarias"] = df_final["fuente_comisarias"].fillna(
    "SIN MATCH CON ANEXO 3 - COMISARIAS 2020"
)

# Comisarías por 100 mil habitantes
if "poblacion" in df_final.columns:
    df_final["comisarias_por_100k_ref_2020"] = np.where(
        df_final["poblacion"].notna()
        & (df_final["poblacion"] > 0)
        & (df_final["n_comisarias_ref_2020"] > 0),
        round((df_final["n_comisarias_ref_2020"] / df_final["poblacion"]) * 100000, 4),
        np.nan
    )
else:
    df_final["comisarias_por_100k_ref_2020"] = np.nan

# Denuncias por comisaría
df_final["denuncias_por_comisaria_ref_2020"] = np.where(
    df_final["n_comisarias_ref_2020"] > 0,
    round(df_final["cantidad"] / df_final["n_comisarias_ref_2020"], 4),
    np.nan
)

# ============================================================
# 14. QUITAR COLUMNAS KEY TEMPORALES
# ============================================================

df_final = df_final.drop(
    columns=[
        "departamento_key",
        "provincia_key",
        "distrito_key"
    ],
    errors="ignore"
)

# ============================================================
# 15. ORDENAR COLUMNAS
# ============================================================

columnas_prioritarias = [
    "anio",
    "mes",
    "fecha",
    "anio_mes",
    "trimestre",
    "ubigeo",
    "departamento",
    "provincia",
    "distrito",
    "poblacion",
    "tasa_denuncias_100k",

    "n_comisarias_ref_2020",
    "comisarias_por_100k_ref_2020",
    "denuncias_por_comisaria_ref_2020",
    "n_comisarias_rurales_ref_2020",
    "n_comisarias_sectoriales_ref_2020",
    "n_comisarias_zonales_ref_2020",
    "match_comisarias_ref_2020",
    "anio_fuente_comisarias",
    "fuente_comisarias",
    "latitud_promedio_comisaria_ref_2020",
    "longitud_promedio_comisaria_ref_2020",

    "n_policias_ref_2025",
    "policias_por_100k_ref_2025",
    "denuncias_por_policia_ref_2025",
    "match_policias",

    "dist_emergencia",
    "tipo_analisis",
    "nivel_detalle",
    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",
    "cantidad",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria",
    "participacion_en_delitos_departamento_pct",
    "participacion_misma_categoria_departamento_pct",
    "tasa_departamento_delitos_100k_referencial",
    "score_completitud"
]

columnas_existentes = [c for c in columnas_prioritarias if c in df_final.columns]
columnas_restantes = [c for c in df_final.columns if c not in columnas_existentes]

df_final = df_final[columnas_existentes + columnas_restantes]

# ============================================================
# 16. EXPORTAR DATA FINAL
# ============================================================

df_final.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 17. QA DEL MERGE
# ============================================================

qa = [
    {"metrica": "archivo_sidpol_usado", "valor": archivo_sidpol.name},
    {"metrica": "archivo_comisarias_usado", "valor": archivo_comisarias.name},
    {"metrica": "hoja_comisarias_usada", "valor": hoja_elegida},
    {"metrica": "filas_sidpol_final", "valor": len(df_final)},
    {"metrica": "comisarias_unicas_anexo3", "valor": df_com["comisaria"].nunique()},
    {"metrica": "comisarias_con_match_ubigeo", "valor": int(df_com["ubigeo"].notna().sum())},
    {"metrica": "comisarias_sin_match_ubigeo", "valor": int(df_com["ubigeo"].isna().sum())},
    {"metrica": "distritos_con_comisarias_anexo3_match", "valor": df_comisarias_distrito["ubigeo"].nunique()},
    {"metrica": "total_comisarias_contadas_con_match", "valor": int(df_comisarias_distrito["n_comisarias_ref_2020"].sum())},
    {"metrica": "filas_con_match_comisarias", "valor": int((df_final["match_comisarias_ref_2020"] == "SI").sum())},
    {"metrica": "filas_sin_match_comisarias", "valor": int((df_final["match_comisarias_ref_2020"] == "NO").sum())},
    {"metrica": "porcentaje_filas_con_match_comisarias", "valor": round((df_final["match_comisarias_ref_2020"] == "SI").mean() * 100, 2)}
]

df_qa = pd.DataFrame(qa)

df_qa.to_csv(
    archivo_qa,
    index=False,
    encoding="utf-8-sig"
)

print("MERGE CON COMISARÍAS 2020 POR NOMBRE TERMINADO")
print(f"Archivo final creado: {archivo_salida.name}")
print(f"Tabla comisarías por distrito creada: {archivo_comisarias_distrito.name}")
print(f"QA creado: {archivo_qa.name}")
print(f"Archivo de comisarías sin match creado: {archivo_no_match.name}")
print(f"Comisarías únicas Anexo 3: {df_com['comisaria'].nunique():,}")
print(f"Comisarías con match UBIGEO SIDPOL: {int(df_com['ubigeo'].notna().sum()):,}")
print(f"Comisarías sin match UBIGEO SIDPOL: {int(df_com['ubigeo'].isna().sum()):,}")
print(f"Total comisarías contadas con match: {int(df_comisarias_distrito['n_comisarias_ref_2020'].sum()):,}")
print(f"Filas SIDPOL con match: {(df_final['match_comisarias_ref_2020'] == 'SI').sum():,}")
print(f"Filas SIDPOL sin match: {(df_final['match_comisarias_ref_2020'] == 'NO').sum():,}")
print(f"% match SIDPOL: {round((df_final['match_comisarias_ref_2020'] == 'SI').mean() * 100, 2)}%")

os.startfile(archivo_salida)